In [5]:
# -*- coding: utf-8 -*-
"""
gptFromScratch_on_imdb.py

A compact, single-file, pedagogical GPT implementation with configurable pretraining dataset.
Key features:
- Configurable pretraining dataset via Config.pretraining_dataset ("stanfordnlp/imdb" or "toy").
- Tokenizer built with huge model_max_length to avoid warnings; we control length via chunking.
- Concatenate+chunk language-modeling dataset creation on CPU with optional token caps for fast demos.
- DataLoader tuned for throughput (num_workers, pin_memory, persistent_workers) and moving to device per batch.
- "pedagogical_mode" forward hooks to print shapes once on first pass.
- Optional fine-tuning section: "Fine-tuning: Coldplay lyrics".
- No Colab-only dependencies; character-level tokenizer section removed.
- Benchmarking of model for quality estimation

### GPT2-124M
| Parameter      | Value   | Description                 |
|----------------|---------|-----------------------------|
| `vocab_size`   | 50257   | Vocabulary size             |
| `context_length`| 1024    | Context length              |
| `emb_dim`      | 768     | Embedding dimension         |
| `n_heads`      | 12      | Number of attention heads   |
| `n_layers`     | 12      | Number of layers            |
| `drop_rate`    | 0.1     | Dropout rate                |
| `qkv_bias`     | False   | Query-Key-Value bias        |

We'll use somewhat smaller numbers here
"""

# ========== 0) Imports ==========
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional, List, Tuple
from datasets import load_dataset
import math
import random
import logging
import os

import torch
from torch import nn # The foundation class for Neural Networks
from torch.utils.data import Dataset, DataLoader, TensorDataset

try:
    from transformers import AutoTokenizer  # optional but recommended
except Exception:
    AutoTokenizer = None

from datasets import load_dataset

# (optional) Hugging Face datasets for IMDB and Coldplay demos
try:
    import datasets as hf_datasets
except Exception:
    hf_datasets = None

import warnings
warnings.filterwarnings(
    "ignore",
    message=r"A NumPy version >=1\.16\.5 and <1\.23\.0 is required",
    category=UserWarning,
    module="scipy"
)

# ========== 1) Config & Utilities ==========

@dataclass
class Config:
    # Dataset control
    pretraining_dataset: str = "alpaca" # עדכנו לשם הדאטה-סט של ההוראות
    imdb_val_fraction: float = 0.05

    # Data / Tokenizer
    tokenizer_name: str = "gpt2"
    context_length: int = 1024 # חלון ההקשר המקורי של המודל gpt2-medium

    # הגדרות אורך (None אומר שהקוד יעבור על כל הדאטה-סט במקום לעצור באמצע)
    max_train_tokens: Optional[int] = None
    max_val_tokens:   Optional[int] = None

    # Model - המפרט המדויק עבור ~350M פרמטרים (gpt2-medium)
    vocab_size: int = 50257
    embed_dim: int = 1024
    n_heads: int = 16
    n_layers: int = 24
    dropout: float = 0.1
    qkv_bias: bool = True # קריטי: חייב להיות True כדי להתאים למשקולות של HuggingFace

    # Training - הגדרות שמותאמות לאימון Fine-Tuning מלא
    batch_size: int = 1 # הערה למטה לגבי הערך הזה
    max_steps: int = 5000 # מספר הצעדים לאימון (ניתן לשנות בהתאם לזמן הריצה הרצוי)
    lr: float = 5e-5 # קצב למידה נמוך ועדין, מתאים ל-Fine-Tuning
    weight_decay: float = 0.1
    warmup_steps: int = 100
    grad_clip: Optional[float] = 1.0

    # Pedagogical & System
    seed: int = 1337
    pedagogical_mode: bool = False # שינינו ל-False כדי למנוע הצפת הדפסות של מימדי הטנזורים
    plot_curves: bool = False
    device: Optional[str] = None
    
def get_device(cfg: Config) -> torch.device:
    if cfg.device is not None:
        return torch.device(cfg.device)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# ========== 2) Tokenization & Data ==========

def build_tokenizer(cfg: Config):
    """
    Builds/loads a tokenizer. If transformers isn't available, a simple whitespace tokenizer fallback is used.
    We set a huge model_max_length so the tokenizer doesn't warn; we control lengths via chunking.
    """
    if AutoTokenizer is None:
        logging.warning("transformers not available; using a simple whitespace tokenizer fallback.")
        class SimpleTok:
            bos_token_id = 0
            eos_token_id = 1
            pad_token_id = 1
            vocab = {"<BOS>":0, "<PAD>":1}
            def __init__(self): pass
            def __call__(self, texts, return_tensors=None, padding=None, truncation=None, max_length=None, add_special_tokens=True, return_attention_mask=False):
                ids = []
                for t in texts:
                    toks = t.strip().split()
                    arr = [self.bos_token_id] + [hash(w)%10000+2 for w in toks]
                    if max_length and padding == "max_length":
                        arr = arr[:max_length]
                        arr = arr + [self.pad_token_id]*(max_length-len(arr))
                    ids.append(arr)
                return {"input_ids": torch.tensor(ids, dtype=torch.long)}
            @property
            def vocab_size(self): return 10002
            @property
            def pad_token(self): return "<PAD>"
            @property
            def eos_token(self): return "<PAD>"
            @pad_token.setter
            def pad_token(self, v): pass
        tok = SimpleTok()
    else:
        # Large cap to avoid model_max_length warnings; we manage length in chunking
        tok = AutoTokenizer.from_pretrained(cfg.tokenizer_name, model_max_length=10**9)
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
    return tok

def make_toy_corpus() -> List[str]:
    # A tiny toy corpus to keep the demo runnable in minutes.
    return [
        "Transformers use self attention to mix information across positions.",
        "Attention allows each token to look at previous tokens and compute a context aware representation.",
        "Language modeling trains a network to predict the next token from previous ones.",
        "Mini GPTs are great for teaching and for sanity checking ideas before training large models.",
    ]

def encode_texts(texts: List[str], tok, max_len: int, device: torch.device) -> torch.Tensor:
    out = tok(texts, return_tensors="pt", padding="max_length", truncation=True, max_length=max_len)
    return out["input_ids"].to(device)

def build_dataloaders(cfg: Config, tok, device: torch.device) -> Tuple[DataLoader, DataLoader]:
    """
    Returns train/val loaders for Instruction Fine-Tuning (SFT)
    with strict loss masking on the user prompt and padding.
    """
    # 1. טעינת דאטה-סט של הוראות (Instruction Dataset) 
    # נשתמש ב-Alpaca כדוגמה קלאסית למשימות instruction-following
    raw = load_dataset("yahma/alpaca-cleaned")
    
    # יצירת פיצול train/val (נשתמש באותו יחס שהיה מוגדר בקונפיגורציה) [cite: 93, 94]
    split = raw["train"].train_test_split(test_size=cfg.imdb_val_fraction, seed=cfg.seed)
    train_split = split["train"]
    val_split = split["test"]

    # הערך שאומר ל-PyTorch להתעלם ממנו בחישוב ה-Cross Entropy Loss 
    IGNORE_INDEX = -100 
    
    # אנחנו צריכים +1 באורך המקסימלי, כי בסוף נחתוך את המערך ליצירת x ו-y (הזזה של 1) [cite: 90]
    max_len = cfg.context_length + 1 

    def process_sft_data(dataset, max_samples: Optional[int] = None):
        x_list, y_list = [], []
        
        for i, row in enumerate(dataset):
            if max_samples and i >= max_samples:
                break
                
            # 2. הגדרת תבנית הפרומפט (Prompt Format) 
            inst = row.get("instruction", "")
            inp = row.get("input", "")
            out = row.get("output", "")
            
            # אם יש קלט נוסף (input), נכניס גם אותו. אחרת, רק הוראה.
            if inp:
                prompt_text = f"User: {inst}\nInput: {inp}\nAssistant: "
            else:
                prompt_text = f"User: {inst}\nAssistant: "
                
            # הטקסט המלא שייכנס למודל כולל את התשובה וטוקן סיום
            full_text = prompt_text + out + (tok.eos_token if tok.eos_token else "")

            # טוקניזציה נפרדת רק לפרומפט - כדי לדעת איפה בדיוק מתחילה התשובה
            prompt_ids = tok(prompt_text, add_special_tokens=True).input_ids
            prompt_len = len(prompt_ids)

            # טוקניזציה לטקסט המלא עם ריפוד (Padding) או חיתוך (Truncation) ל-max_len
            encoded = tok(
                full_text,
                add_special_tokens=True,
                max_length=max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )
            
            input_ids = encoded.input_ids[0]
            
            # 3. יישום מיסוך ל-Loss (Loss Masking) 
            labels = input_ids.clone()
            
            # מסתירים את כל הטוקנים ששייכים לשאלת המשתמש (מאינדקס 0 עד אורך הפרומפט)
            labels[:prompt_len] = IGNORE_INDEX 
            
            # מסתירים גם את כל טוקני הריפוד (Padding) שנוספו בסוף כדי שהמודל לא ילמד לחזות סתם רווחים
            pad_token_id = tok.pad_token_id if tok.pad_token_id is not None else 0
            labels[input_ids == pad_token_id] = IGNORE_INDEX 
            
            # 4. יצירת X ו-Y על ידי הזזה (Shift) בדומה לפונקציית ה-chunker המקורית 
            # X הוא כל הטוקנים חוץ מהאחרון, Y הוא כל הטוקנים (והתוויות) חוץ מהראשון
            x = input_ids[:-1].contiguous()
            y = labels[1:].contiguous()
            
            x_list.append(x)
            y_list.append(y)

        return torch.stack(x_list), torch.stack(y_list)

    # המרה של כמות הטוקנים המקסימלית המותרת (מהקונפיגורציה) לכמות דוגמאות [cite: 75, 93]
    max_train_samples = (cfg.max_train_tokens // max_len) if cfg.max_train_tokens else None
    max_val_samples = (cfg.max_val_tokens // max_len) if cfg.max_val_tokens else None

    # עיבוד הנתונים
    print("Processing training data...")
    x_tr, y_tr = process_sft_data(train_split, max_train_samples)
    print("Processing validation data...")
    x_va, y_va = process_sft_data(val_split, max_val_samples)

    # הגדרות PyTorch עבור ה-DataLoaders [cite: 95]
    is_cuda = (device.type == "cuda") 
    nw = max(1, (os.cpu_count() or 4) // 2) 
    persistent = True if nw > 0 else False 

    train_loader = DataLoader(
        TensorDataset(x_tr, y_tr),
        batch_size=cfg.batch_size,
        shuffle=True,
        drop_last=True, 
        num_workers=nw,
        pin_memory=is_cuda,
        persistent_workers=persistent,
    )
    val_loader = DataLoader(
        TensorDataset(x_va, y_va),
        batch_size=cfg.batch_size,
        shuffle=False,
        drop_last=False, 
        num_workers=nw,
        pin_memory=is_cuda,
        persistent_workers=persistent,
    )
    
    return train_loader, val_loader

    # --------- Fallback: toy corpus ----------
    texts = make_toy_corpus()
    n_train = max(2, int(0.75*len(texts)))
    train_ids = encode_texts(texts[:n_train], tok, cfg.context_length, device)
    val_ids   = encode_texts(texts[n_train:], tok, cfg.context_length, device)
    x_tr = train_ids[:, :-1].contiguous()
    y_tr = train_ids[:,  1:].contiguous()
    x_va = val_ids[:, :-1].contiguous()
    y_va = val_ids[:,  1:].contiguous()
    train_loader = DataLoader(TensorDataset(x_tr, y_tr), batch_size=cfg.batch_size, shuffle=True)
    val_loader   = DataLoader(TensorDataset(x_va, y_va), batch_size=cfg.batch_size, shuffle=False)
    return train_loader, val_loader

# ========== 3) Basic Architecture - Model Blocks ==========

class SelfAttention(nn.Module):
    """
    Single-head **masked** (causal) self-attention.

    - Takes token embeddings x of shape (B, T, C)
    - Projects them to queries, keys, values of shape (B, T, D_head)
    - Computes scaled dot-product attention with a causal mask
    - Returns attended values of shape (B, T, D_head)
    """
    def __init__(self, embed_dim: int, head_dim: int, bias: bool = False, dropout: float = 0.1):
        super().__init__()
        self.head_dim = head_dim

        # Learnable projections from model dimension -> head dimension
        self.w_query = nn.Linear(embed_dim, head_dim, bias=bias)
        self.w_key   = nn.Linear(embed_dim, head_dim, bias=bias)
        self.w_value = nn.Linear(embed_dim, head_dim, bias=bias)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, T, C) token embeddings
        returns: (B, T, D_head) attended features for this head
        """
        B, T, _ = x.size()

        # 1) Project to q, k, v
        q = self.w_query(x)  # (B, T, D_head)
        k = self.w_key(x)    # (B, T, D_head)
        v = self.w_value(x)  # (B, T, D_head)

        # 2) Scaled dot-product attention scores
        #    scores[b, t_q, t_k] = <q[b, t_q], k[b, t_k]> / sqrt(D_head)
        scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)  # (B, T, T)

        # 3) Causal mask: prevent looking into the future
        #    mask[t_q, t_k] = True if t_k > t_q
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-1e10"))

        # 4) Convert scores to probabilities
        attn = scores.softmax(dim=-1)  # (B, T, T)
        attn = self.dropout(attn)

        # 5) Weighted sum of values
        out = attn @ v                 # (B, T, D_head)
        return out


class MultiHeadAttention(nn.Module):
    """
    Multi-head masked self-attention built from SelfAttention heads.

    - Splits the model dimension C into H heads of size D_head = C / H
    - Each head runs its own SelfAttention (independent projections)
    - Concatenates head outputs back to (B, T, C)
    - Optional final linear projection can be added if desired
    """
    def __init__(self, embed_dim: int, num_heads: int, bias: bool = False, dropout: float = 0.1, max_ctx: int = 2048):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads

        # List of independent attention heads
        self.heads = nn.ModuleList([
            SelfAttention(embed_dim=embed_dim, head_dim=self.head_dim, bias=bias, dropout=dropout)
            for _ in range(num_heads)
        ])

        # Final projection back to embed_dim (keeps interface consistent)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (B, T, C) where C = embed_dim
        returns: (B, T, C)
        """
        # Run each head on the same input x
        head_outputs = [head(x) for head in self.heads]     # list of (B, T, D_head)
        # Concatenate along the feature dimension to recover (B, T, C)
        concat = torch.cat(head_outputs, dim=-1)            # (B, T, num_heads * D_head) = (B, T, C)
        # Optional output projection mixes head information
        return self.out_proj(concat)

class FeedForward(nn.Module):
    def __init__(self, embed_dim: int, hidden_dim: int, dropout: float=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class DecoderBlock(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, dropout: float, qkv_bias: bool, max_ctx: int = 2048):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads, bias=qkv_bias, dropout=dropout, max_ctx=max_ctx)
        self.ffn  = FeedForward(embed_dim, hidden_dim=4*embed_dim, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class GPT(nn.Module):
    """
    Minimal GPT-style decoder-only transformer.
    """
    def __init__(self, vocab_size: int, context_length: int, embed_dim: int, n_layers: int,
                 n_heads: int, dropout: float=0.0, qkv_bias: bool=False):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            DecoderBlock(embed_dim, n_heads, dropout, qkv_bias, max_ctx=context_length) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.context_length = context_length
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m: nn.Module):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device, dtype=torch.long).unsqueeze(0)  # (1,T)
        x = self.tok_emb(idx) + self.pos_emb(pos)  # (B,T,C)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (B,T,vocab)
        return logits

# ========== 4) Shape-print hooks ==========

class _PrintOnce:
    def __init__(self): self.done = False
    def should(self):
        if self.done: return False
        self.done = True
        return True

def register_pedagogical_hooks(model: nn.Module, enabled: bool=True) -> List[torch.utils.hooks.RemovableHandle]:
    """
    If enabled, prints shapes at key points during the first forward pass.
    """
    handles = []
    if not enabled:
        return handles

    printer = _PrintOnce()

    def make_hook(name):
        def hook(mod, inp, out):
            if printer.should():
                def shape(x):
                    return tuple(x.shape) if isinstance(x, torch.Tensor) else str(type(x))
                in_shapes = [shape(t) for t in (inp if isinstance(inp, (list,tuple)) else [inp])]
                out_shape = shape(out)
                print(f"[pedagogical] {name}: input {in_shapes} -> output {out_shape}")
        return hook

    for i, blk in enumerate(model.blocks):
        handles.append(blk.ln1.register_forward_hook(make_hook(f"Block{i}.LN1")))
        handles.append(blk.attn.register_forward_hook(make_hook(f"Block{i}.MHA")))
        handles.append(blk.ln2.register_forward_hook(make_hook(f"Block{i}.LN2")))
        handles.append(blk.ffn.register_forward_hook(make_hook(f"Block{i}.FFN")))
    handles.append(model.head.register_forward_hook(make_hook("LMHead")))
    return handles

# ========== 5) Loss, Evaluation, Scheduler ==========

def cross_entropy_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    # logits: (B,T,V), targets: (B,T)
    B, T, V = logits.shape
    return nn.functional.cross_entropy(logits.view(B*T, V), targets.view(B*T))

class CosineWithWarmup:
    # Lightweight LR scheduler: linear warmup then cosine decay to 10% of initial LR.
    def __init__(self, optimizer: torch.optim.Optimizer, warmup_steps: int, max_steps: int, base_lr: float):
        self.opt = optimizer
        self.warmup = max(1, warmup_steps)
        self.max_steps = max_steps
        self.base_lr = base_lr
        self.t = 0

    def step(self):
        self.t += 1
        if self.t < self.warmup:
            lr = self.base_lr * self.t / self.warmup
        else:
            # cosine from base_lr -> 0.1*base_lr
            progress = (self.t - self.warmup) / max(1, (self.max_steps - self.warmup))
            min_lr = 0.1 * self.base_lr
            lr = min_lr + 0.5*(self.base_lr - min_lr)*(1 + math.cos(math.pi * progress))
        for g in self.opt.param_groups:
            g["lr"] = lr
        return lr

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    losses = []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        logits = model(x)
        loss = cross_entropy_loss(logits, y)
        losses.append(loss.item())
    model.train()
    return float(sum(losses) / max(1, len(losses)))

# # ==========================================
#     # Memory Optimization: הקפאת השכבות התחתונות
#     # ==========================================
#     # 1. מקפיאים את שכבות ההטמעה (Embeddings)
#     for param in model.tok_emb.parameters():
#         param.requires_grad = False
#     for param in model.pos_emb.parameters():
#         param.requires_grad = False
        
#     # 2. מקפיאים את 18 השכבות הראשונות (מתוך 24)
#     for i in range(18):
#         for param in model.blocks[i].parameters():
#             param.requires_grad = False
            
#     # מדפיסים כמה פרמטרים באמת מתאמנים עכשיו
#     trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
#     logging.info(f"Trainable parameters after freezing: {trainable:,}")
#     # ==========================================
    
# ========== 6) Training & Generation ==========

def train(cfg: Config, model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, device: torch.device):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = CosineWithWarmup(opt, cfg.warmup_steps, cfg.max_steps, cfg.lr)

    hooks = register_pedagogical_hooks(model, cfg.pedagogical_mode)

    step = 0
    while step < cfg.max_steps:
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            logits = model(xb)
            loss = cross_entropy_loss(logits, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            if cfg.grad_clip:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
            lr = sched.step()

            if step % 10 == 0:
                val_loss = evaluate(model, val_loader, device)
                logging.info(f"step {step:4d} | train_loss {loss.item():.4f} | val_loss {val_loss:.4f} | lr {lr:.2e}")
            step += 1
            if step >= cfg.max_steps:
                break

    for h in hooks:
        try:
            h.remove()
        except Exception:
            pass

@torch.no_grad()
def generate(model: nn.Module, idx: torch.Tensor, max_new_tokens: int, context_length: int,
             temperature: float=1.0, top_k: Optional[int]=None, eos_id: Optional[int]=None) -> torch.Tensor:
    """
    Autoregressive generation from context idx (B,T).
    Stops early if eos_id is generated.
    """
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_length:]
        logits = model(idx_cond)[:, -1, :] / max(1e-8, temperature)
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[..., [-1]]] = -float("inf")
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        # מוסיפים את הטוקן החדש לרצף
        idx = torch.cat([idx, next_token], dim=1)
        
        # תנאי עצירה: אם המודל בחר את טוקן הסיום, עוצרים את היצירה!
        if eos_id is not None and next_token.item() == eos_id:
            break
            
    model.train()
    return idx
    
# ==========

def load_hf_weights_into_custom_gpt(model: nn.Module, model_name: str = "gpt2-medium"):
    from transformers import GPT2LMHeadModel
    print(f"\nDownloading pretrained weights for {model_name}...")
    hf_model = GPT2LMHeadModel.from_pretrained(model_name)
    sd_hf = hf_model.state_dict()
    sd_custom = model.state_dict()

    def copy_w(custom_key, hf_key, transpose=False):
        if hf_key in sd_hf:
            w = sd_hf[hf_key]
            if transpose:
                w = w.t() # המרה מ-Conv1D ל-Linear
            if custom_key in sd_custom:
                if sd_custom[custom_key].shape == w.shape:
                    sd_custom[custom_key].copy_(w)
                else:
                    print(f"Shape mismatch! Custom {custom_key}: {sd_custom[custom_key].shape}, HF {hf_key}: {w.shape}")

    # Embeddings
    copy_w("tok_emb.weight", "transformer.wte.weight")
    copy_w("pos_emb.weight", "transformer.wpe.weight")

    # Blocks
    for i in range(len(model.blocks)):
        # LayerNorms
        copy_w(f"blocks.{i}.ln1.weight", f"transformer.h.{i}.ln_1.weight")
        copy_w(f"blocks.{i}.ln1.bias", f"transformer.h.{i}.ln_1.bias")
        copy_w(f"blocks.{i}.ln2.weight", f"transformer.h.{i}.ln_2.weight")
        copy_w(f"blocks.{i}.ln2.bias", f"transformer.h.{i}.ln_2.bias")

        # Attention QKV mapping (HF to custom ModuleList of heads)
        hf_qkv_w = sd_hf[f"transformer.h.{i}.attn.c_attn.weight"].t() # [3*dim, dim]
        hf_qkv_b = sd_hf[f"transformer.h.{i}.attn.c_attn.bias"]       # [3*dim]
        
        dim = model.blocks[i].attn.embed_dim
        num_heads = model.blocks[i].attn.num_heads
        head_dim = model.blocks[i].attn.head_dim

        # פיצול המטריצה הגדולה ל-Q, K, V
        hf_Q_w, hf_K_w, hf_V_w = hf_qkv_w.split(dim, dim=0) 
        hf_Q_b, hf_K_b, hf_V_b = hf_qkv_b.split(dim, dim=0) 

        # חלוקת המשקולות לכל ראש בנפרד (כפי שמוגדר במחלקת ה-MultiHeadAttention שלך)
        for h in range(num_heads):
            start = h * head_dim
            end = (h + 1) * head_dim
            
            sd_custom[f"blocks.{i}.attn.heads.{h}.w_query.weight"].copy_(hf_Q_w[start:end, :])
            sd_custom[f"blocks.{i}.attn.heads.{h}.w_key.weight"].copy_(hf_K_w[start:end, :])
            sd_custom[f"blocks.{i}.attn.heads.{h}.w_value.weight"].copy_(hf_V_w[start:end, :])
            
            if model.blocks[i].attn.heads[h].w_query.bias is not None:
                sd_custom[f"blocks.{i}.attn.heads.{h}.w_query.bias"].copy_(hf_Q_b[start:end])
                sd_custom[f"blocks.{i}.attn.heads.{h}.w_key.bias"].copy_(hf_K_b[start:end])
                sd_custom[f"blocks.{i}.attn.heads.{h}.w_value.bias"].copy_(hf_V_b[start:end])

        # Attention Projection
        copy_w(f"blocks.{i}.attn.out_proj.weight", f"transformer.h.{i}.attn.c_proj.weight", transpose=True)
        copy_w(f"blocks.{i}.attn.out_proj.bias", f"transformer.h.{i}.attn.c_proj.bias")

        # Feed Forward Network
        copy_w(f"blocks.{i}.ffn.net.0.weight", f"transformer.h.{i}.mlp.c_fc.weight", transpose=True)
        copy_w(f"blocks.{i}.ffn.net.0.bias", f"transformer.h.{i}.mlp.c_fc.bias")
        copy_w(f"blocks.{i}.ffn.net.2.weight", f"transformer.h.{i}.mlp.c_proj.weight", transpose=True)
        copy_w(f"blocks.{i}.ffn.net.2.bias", f"transformer.h.{i}.mlp.c_proj.bias")

    # Final LayerNorm & LM Head
    copy_w("ln_f.weight", "transformer.ln_f.weight")
    copy_w("ln_f.bias", "transformer.ln_f.bias")
    sd_custom["head.weight"].copy_(sd_hf["transformer.wte.weight"])

    print("Pretrained weights successfully injected into your custom model!")

    
# ========== 7) Main: the build the entire flow, with (optional) fine-tune, generate ==========

def main():
    logging.basicConfig(level=logging.INFO, format="%(message)s")

    cfg = Config()
    set_seed(cfg.seed)
    device = get_device(cfg)
    logging.info(f"Using device: {device}")

    # Tokenizer
    tok = build_tokenizer(cfg)
    vocab_size = getattr(tok, "vocab_size", cfg.vocab_size)
    cfg.vocab_size = vocab_size
    logging.info(f"Tokenizer vocab size: {cfg.vocab_size}")

    # Data
    train_loader, val_loader = build_dataloaders(cfg, tok, device)

    # Model
    model = GPT(
        vocab_size=cfg.vocab_size,
        context_length=cfg.context_length,
        embed_dim=cfg.embed_dim,
        n_layers=cfg.n_layers,
        n_heads=cfg.n_heads,
        dropout=cfg.dropout,
        qkv_bias=cfg.qkv_bias,
    ).to(device)
    load_hf_weights_into_custom_gpt(model, "gpt2-medium")
    
    # Pretraining
    title = "IMDB" if cfg.pretraining_dataset.lower() == "stanfordnlp/imdb" else "Instruction SFT"
    logging.info(f"=== Pretraining ({title}) ===")
    train(cfg, model, train_loader, val_loader, device)

    checkpoint_path = "gpt_sft_model.pt"
    torch.save(model.state_dict(), checkpoint_path)
    logging.info(f"Model successfully saved to {checkpoint_path}")

 
# ---------- Generation demo ----------
    logging.info("\n=== Generation demo ===")
    
    # 1. השאלה שלנו
    instruction = "What is the capital of France?"
    
    # 2. עטיפת השאלה בתבנית המדויקת שהמודל ראה באימון
    # שים לב שאנחנו משאירים את ה-"Assistant: " פתוח כדי שהמודל ישלים אותו
    prompt = f"User: {instruction}\nAssistant: "
    
    # 3. טוקניזציה (כאן לא צריך padding כי אנחנו מכניסים רק שאלה אחת, לא אצווה שלמה)
    ctx = tok(prompt, return_tensors="pt").input_ids.to(device)
    
    # 4. יצירת התשובה (נעביר את ה-eos_token_id כדי שידע מתי לעצור)
    eos_id = tok.eos_token_id
    out = generate(
        model, 
        ctx, 
        max_new_tokens=50, 
        context_length=cfg.context_length, 
        temperature=0.7,  # טמפרטורה נמוכה קצת עדיפה לתשובות עובדתיות
        top_k=50,
        eos_id=eos_id
    )
    
    # 5. פענוח והדפסה
    if hasattr(tok, "decode"):
        try:
            # אנחנו מפענחים רק את מה שהמודל ייצר (מדלגים על הפרומפט המקורי)
            generated_tokens = out[0].tolist()
            text = tok.decode(generated_tokens, skip_special_tokens=False)
        except Exception:
            text = str(out[0].tolist())
    else:
        text = str(out[0].tolist())
        
    print("\n--- Model Output ---")
    print(text)
    print("--------------------")

 

# if __name__ == "__main__":
#     main()

In [12]:
import torch

# Instantiate the configuration object
cfg = Config()

# Check for available device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Create an empty model using the exact parameter names your GPT class expects
model = GPT(
    vocab_size=cfg.vocab_size,
    context_length=cfg.context_length,
    embed_dim=cfg.embed_dim,
    n_heads=cfg.n_heads,      # Fixed: Changed left side to n_heads
    n_layers=cfg.n_layers,    # Fixed: Changed left side to n_layers
    dropout=cfg.dropout,      # Fixed: Changed left side to dropout
    qkv_bias=cfg.qkv_bias,
).to(device)

# 2. Inject the trained weights from the saved file
model.load_state_dict(torch.load("gpt_sft_model.pt", map_location=device))

# 3. Set the model to evaluation mode
model.eval()

print("Trained model successfully loaded from disk! 🚀")

/tmp/ipykernel_33580/3584796095.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("gpt_sft_model.pt", map_location=device))


Trained model successfully loaded from disk! 🚀


In [14]:
from transformers import GPT2Tokenizer

# Load the tokenizer from Hugging Face
tok = GPT2Tokenizer.from_pretrained("gpt2-medium")
tok.pad_token = tok.eos_token
print("Tokenizer loaded successfully! 📝")

HTTP Request: HEAD https://huggingface.co/gpt2-medium/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/gpt2-medium/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

HTTP Request: GET https://huggingface.co/api/models/gpt2-medium/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2-medium/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/gpt2-medium/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2-medium/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/gpt2-medium/resolve/main/vocab.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/gpt2-medium/resolve/main/vocab.json "HTTP/1.1 200 OK"


vocab.json: 0.00B [00:00, ?B/s]

HTTP Request: HEAD https://huggingface.co/gpt2-medium/resolve/main/merges.txt "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/gpt2-medium/resolve/main/merges.txt "HTTP/1.1 200 OK"


merges.txt: 0.00B [00:00, ?B/s]

HTTP Request: HEAD https://huggingface.co/gpt2-medium/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/gpt2-medium/resolve/main/special_tokens_map.json "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://huggingface.co/gpt2-medium/resolve/main/tokenizer.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/gpt2-medium/resolve/main/tokenizer.json "HTTP/1.1 200 OK"


tokenizer.json: 0.00B [00:00, ?B/s]

HTTP Request: HEAD https://huggingface.co/gpt2-medium/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


Tokenizer loaded successfully! 📝


In [16]:
import torch
from datasets import load_dataset
from tqdm import tqdm

# Ensure model is in evaluation mode (disables dropout etc.)
model.eval()

def generate_answer(prompt, model, tokenizer, device, max_new_tokens=30):
    """
    Generates a response using the trained model, applying the SFT prompt format.
    """
    # Format the prompt exactly like our instruction training data
    full_prompt = f"User: {prompt}\nAssistant:\n"
    
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_ids = inputs.input_ids
    
    # Autoregressive generation loop
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids)
            # Take the logits for the last token
            next_token_logits = logits[:, -1, :]
            # Greedy decoding: pick the most probable next token
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            
            # Append to the sequence
            input_ids = torch.cat([input_ids, next_token], dim=1)
            
            # Stop if End-Of-Sequence token is generated
            if next_token.item() == tokenizer.eos_token_id:
                break
                
    # Extract only the newly generated tokens (ignore the prompt)
    generated_ids = input_ids[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response.strip()

def evaluate_squad(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the SQuAD validation dataset (Reading Comprehension).
    """
    print("\n--- Evaluating SQuAD ---")
    # Added trust_remote_code=True to bypass security block
    dataset = load_dataset("squad", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        context = item['context']
        question = item['question']
        valid_answers = item['answers']['text'] # List of acceptable answers
        
        prompt = f"Context: {context}\nQuestion: {question}\nAnswer shortly."
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        # Check if any of the valid answers are present in the model's output
        is_correct = any(ans.lower() in prediction.lower() for ans in valid_answers)
        if is_correct:
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    print(f"SQuAD Accuracy (Subset): {accuracy:.2f}%")
    return accuracy

def evaluate_babi(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the bAbI Task 1 dataset (Basic Deduction).
    """
    print("\n--- Evaluating bAbI (Task 1) ---")
    # Added trust_remote_code=True to bypass security block
    dataset = load_dataset("babi_qa", type="en", split="valid", trust_remote_code=True)
    
    # Filter only Task 1
    task1_data = [item for item in dataset if item['type'] == 'task1']
    num_eval = min(num_samples, len(task1_data))
    correct = 0
    
    for i in tqdm(range(num_eval)):
        item = task1_data[i]
        story = item['story']['text']
        question = item['question']
        expected_answer = item['answer']
        
        # Join the story sentences into a single paragraph
        context = " ".join(story)
        prompt = f"Story: {context}\nQuestion: {question}\nAnswer in one word."
        
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        if expected_answer.lower() in prediction.lower():
            correct += 1
            
    accuracy = (correct / num_eval) * 100
    print(f"bAbI Task 1 Accuracy (Subset): {accuracy:.2f}%")
    return accuracy

def evaluate_arc(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the ARC-Easy dataset (Science Multiple Choice).
    """
    print("\n--- Evaluating ARC-Easy ---")
    # Added trust_remote_code=True to bypass security block
    dataset = load_dataset("ai2_arc", "ARC-Easy", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        question = item['question']
        choices = item['choices']['text']
        labels = item['choices']['label']
        correct_label = item['answerKey']
        
        # Format the multiple choice question
        choices_text = "\n".join([f"{label}: {text}" for label, text in zip(labels, choices)])
        prompt = f"Question: {question}\nChoices:\n{choices_text}\nWhich choice is correct? State only the label."
        
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        # Get the actual correct text to also check against it
        correct_idx = labels.index(correct_label)
        correct_text = choices[correct_idx]
        
        # Consider correct if either the label (e.g., 'A') or the text is in the prediction
        if correct_label in prediction or correct_text.lower() in prediction.lower():
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    print(f"ARC-Easy Accuracy (Subset): {accuracy:.2f}%")
    return accuracy

# Run the evaluation suite
if __name__ == "__main__":
    print("Starting Evaluation Suite...")
    squad_acc = evaluate_squad(model, tok, device, num_samples=100)
    babi_acc = evaluate_babi(model, tok, device, num_samples=100)
    arc_acc = evaluate_arc(model, tok, device, num_samples=100)
    
    print("\n==============================")
    print("FINAL EVALUATION RESULTS")
    print("==============================")
    print(f"SQuAD Accuracy:    {squad_acc:.2f}%")
    print(f"bAbI Accuracy:     {babi_acc:.2f}%")
    print(f"ARC-Easy Accuracy: {arc_acc:.2f}%")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'squad' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'squad' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Starting Evaluation Suite...

--- Evaluating SQuAD ---


HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/rajpurkar/squad/7b6d24c440a36b6815f21b70d25016731768db1f/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/squad/squad.py "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggi

SQuAD Accuracy (Subset): 71.00%

--- Evaluating bAbI (Task 1) ---


HTTP Request: HEAD https://huggingface.co/datasets/babi_qa/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/facebook/babi_qa/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/facebook/babi_qa/021d7aeb7307b7856dd0632f92827bc607dc2f1b/README.md "HTTP/1.1 200 OK"


RuntimeError: Dataset scripts are no longer supported, but found babi_qa.py

In [17]:
import torch
from datasets import load_dataset
from tqdm import tqdm

# Ensure model is in evaluation mode (disables dropout etc.)
model.eval()

def generate_answer(prompt, model, tokenizer, device, max_new_tokens=30):
    """
    Generates a response using the trained model, applying the SFT prompt format.
    """
    # Format the prompt exactly like our instruction training data
    full_prompt = f"User: {prompt}\nAssistant:\n"
    
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_ids = inputs.input_ids
    
    # Autoregressive generation loop
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids)
            # Take the logits for the last token
            next_token_logits = logits[:, -1, :]
            # Greedy decoding: pick the most probable next token
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            
            # Append to the sequence
            input_ids = torch.cat([input_ids, next_token], dim=1)
            
            # Stop if End-Of-Sequence token is generated
            if next_token.item() == tokenizer.eos_token_id:
                break
                
    # Extract only the newly generated tokens (ignore the prompt)
    generated_ids = input_ids[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response.strip()

def evaluate_squad(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the SQuAD validation dataset (Reading Comprehension).
    """
    print("\n--- Evaluating SQuAD ---")
    dataset = load_dataset("squad", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        context = item['context']
        question = item['question']
        valid_answers = item['answers']['text'] # List of acceptable answers
        
        prompt = f"Context: {context}\nQuestion: {question}\nAnswer shortly."
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        # Check if any of the valid answers are present in the model's output
        is_correct = any(ans.lower() in prediction.lower() for ans in valid_answers)
        if is_correct:
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    print(f"SQuAD Accuracy (Subset): {accuracy:.2f}%")
    return accuracy

def evaluate_arc(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the ARC-Easy dataset (Science Multiple Choice).
    """
    print("\n--- Evaluating ARC-Easy ---")
    dataset = load_dataset("ai2_arc", "ARC-Easy", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        question = item['question']
        choices = item['choices']['text']
        labels = item['choices']['label']
        correct_label = item['answerKey']
        
        # Format the multiple choice question
        choices_text = "\n".join([f"{label}: {text}" for label, text in zip(labels, choices)])
        prompt = f"Question: {question}\nChoices:\n{choices_text}\nWhich choice is correct? State only the label."
        
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        # Get the actual correct text to also check against it
        correct_idx = labels.index(correct_label)
        correct_text = choices[correct_idx]
        
        # Consider correct if either the label (e.g., 'A') or the text is in the prediction
        if correct_label in prediction or correct_text.lower() in prediction.lower():
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    print(f"ARC-Easy Accuracy (Subset): {accuracy:.2f}%")
    return accuracy

# Run the evaluation suite
if __name__ == "__main__":
    print("Starting Evaluation Suite...")
    squad_acc = evaluate_squad(model, tok, device, num_samples=100)
    arc_acc = evaluate_arc(model, tok, device, num_samples=100)
    
    print("\n==============================")
    print("FINAL EVALUATION RESULTS")
    print("==============================")
    print(f"SQuAD Accuracy:    {squad_acc:.2f}%")
    print(f"ARC-Easy Accuracy: {arc_acc:.2f}%")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'squad' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'squad' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Starting Evaluation Suite...

--- Evaluating SQuAD ---


HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/rajpurkar/squad/7b6d24c440a36b6815f21b70d25016731768db1f/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/squad/squad.py "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggi

SQuAD Accuracy (Subset): 71.00%

--- Evaluating ARC-Easy ---


HTTP Request: HEAD https://huggingface.co/datasets/ai2_arc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/allenai/ai2_arc/210d026faf9955653af8916fad021475a3f00453/README.md "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/allenai/ai2_arc/210d026faf9955653af8916fad021475a3f00453/README.md "HTTP/1.1 200 OK"


...

Generating train split:   0%|          | 0/2251 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2376 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/570 [00:00<?, ? examples/s]

100%|█████████████████████████████████████████| 100/100 [04:55<00:00,  2.96s/it]

ARC-Easy Accuracy (Subset): 80.00%

FINAL EVALUATION RESULTS
SQuAD Accuracy:    71.00%
ARC-Easy Accuracy: 80.00%


In [18]:
import torch

def demonstrate_instruction_following(model, tokenizer, device):
    print("=== Demonstrating Instruction-Following Behavior ===\n")
    
    # מגוון רחב של הוראות כדי להוכיח שהמודל מגיב נכון לסוגים שונים של משימות
    test_prompts = [
        "What is the capital of France?",                   # שאלת ידע בסיסית
        "Write a short poem about a brave cat.",            # בקשת יצירתיות
        "Translate the word 'Hello' to Spanish.",           # משימת תרגום
        "Explain what a black hole is in simple words.",    # בקשת הסבר
        "List three healthy fruits."                        # משימת רשימה
    ]
    
    model.eval()
    
    for i, prompt in enumerate(test_prompts):
        print(f"--- Test {i+1} ---")
        print(f"Prompt (User): {prompt}")
        
        # עטיפת השאלה בפורמט שהמודל למד
        full_prompt = f"User: {prompt}\nAssistant:\n"
        
        inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
        input_ids = inputs.input_ids
        
        # יצירת התשובה
        with torch.no_grad():
            for _ in range(40): # מגבלת טוקנים קצרה רק כדי לראות את כיוון התשובה
                logits = model(input_ids)
                next_token_logits = logits[:, -1, :]
                
                # בוחרים את הטוקן הכי סביר
                next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
                input_ids = torch.cat([input_ids, next_token], dim=1)
                
                if next_token.item() == tokenizer.eos_token_id:
                    break
                    
        # חילוץ התשובה בלבד (ללא הפרומפט המקורי)
        generated_ids = input_ids[0][inputs.input_ids.shape[1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        
        print(f"Model (Assistant):\n{response}\n")

# הרצת ההדגמה
demonstrate_instruction_following(model, tok, device)

=== Demonstrating Instruction-Following Behavior ===

--- Test 1 ---
Prompt (User): What is the capital of France?
Model (Assistant):
The capital of France is Paris. It is the city of the Republic of France, with a population of approximately 1.5 million people. It is located in the city of Paris, which is approximately

--- Test 2 ---
Prompt (User): Write a short poem about a brave cat.
Model (Assistant):
A brave cat,
A brave cat,
A brave cat,
A brave cat,
A brave cat,
A brave cat,
A brave cat,
A brave cat,

--- Test 3 ---
Prompt (User): Translate the word 'Hello' to Spanish.
Model (Assistant):
- Spanish: "Ella"
- Spanish: "Ella"
- Spanish: "Ella"
- Spanish: "Ella"
- Spanish: "Ella"

--- Test 4 ---
Prompt (User): Explain what a black hole is in simple words.
Model (Assistant):
A black hole is a massive, dense object that is surrounded by a dense, rotating disk of matter and energy. It is a vast expanse of space that is filled with a dense, rotating

--- Test 5 ---
Prompt (User): List 

In [19]:
import torch
from transformers import GPT2LMHeadModel

def compare_behavioral_shift(fine_tuned_model, tokenizer, device):
    print("Loading Base GPT-2 Medium for comparison... ⏳")
    # טעינת המודל המקורי מהאינטרנט לצורך ההשוואה
    base_model = GPT2LMHeadModel.from_pretrained("gpt2-medium").to(device)
    base_model.eval()
    fine_tuned_model.eval()
    
    test_prompts = [
        "What is the capital of France?",
        "Explain what a black hole is in simple words.",
        "Translate the word 'Hello' to Spanish."
    ]
    
    print("\n" + "="*50)
    print(" BEHAVIORAL SHIFT COMPARISON")
    print("="*50 + "\n")
    
    for i, prompt in enumerate(test_prompts):
        print(f"🔹 TEST {i+1} 🔹")
        print(f"Prompt (User): {prompt}\n")
        
        full_prompt = f"User: {prompt}\nAssistant:\n"
        inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
        
        # --- 1. Base Model Generation (Text Continuation) ---
        with torch.no_grad():
            base_outputs = base_model.generate(
                inputs.input_ids, 
                max_new_tokens=30, 
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False # Greedy decoding לשם הוגנות בהשוואה
            )
        base_text = tokenizer.decode(base_outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # --- 2. Fine-Tuned Model Generation (Instruction Following) ---
        input_ids = inputs.input_ids
        with torch.no_grad():
            for _ in range(30):
                logits = fine_tuned_model(input_ids)
                next_token_logits = logits[:, -1, :]
                next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
                input_ids = torch.cat([input_ids, next_token], dim=1)
                
                if next_token.item() == tokenizer.eos_token_id:
                    break
                    
        ft_text = tokenizer.decode(input_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # --- Print Results ---
        print("[Base GPT-2 Output] (Notice how it continues the text):")
        print(f"-> {base_text}\n")
        
        print("[Your Fine-Tuned Model Output] (Notice how it answers the prompt):")
        print(f"-> {ft_text}\n")
        print("-" * 50 + "\n")

# הפעלת ההשוואה
compare_behavioral_shift(model, tok, device)

Loading Base GPT-2 Medium for comparison... ⏳


HTTP Request: HEAD https://huggingface.co/gpt2-medium/resolve/main/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/gpt2-medium/resolve/main/model.safetensors "HTTP/1.1 302 Found"


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

HTTP Request: HEAD https://huggingface.co/gpt2-medium/resolve/main/generation_config.json "HTTP/1.1 200 OK"
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



 BEHAVIORAL SHIFT COMPARISON

🔹 TEST 1 🔹
Prompt (User): What is the capital of France?

[Base GPT-2 Output] (Notice how it continues the text):
-> Assistant:
Assistant:
Assistant:
Assistant:
Assistant:
Assistant:
Assistant:
Assistant:
Assistant:
Assistant:

[Your Fine-Tuned Model Output] (Notice how it answers the prompt):
-> The capital of France is Paris. It is the city of the Republic of France, with a population of approximately 1.5 million people. It is

--------------------------------------------------

🔹 TEST 2 🔹
Prompt (User): Explain what a black hole is in simple words.

[Base GPT-2 Output] (Notice how it continues the text):
-> Black holes are the centers of all matter in the universe. They are the centers of all matter in the universe.
Assistant:
Black holes are

[Your Fine-Tuned Model Output] (Notice how it answers the prompt):
-> A black hole is a massive, dense object that is surrounded by a dense, rotating disk of matter and energy. It is a vast expanse

-------------

In [26]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================================
# 1. Update Config (Run this first to ensure the new params exist)
# ============================================================================
cfg = Config()
cfg.num_experts = 4
cfg.top_k = 2

# ============================================================================
# 3. UPGRADED ARCHITECTURE: Custom MoE Components
# ============================================================================

class Expert(nn.Module):
    """
    Acts as a single expert. Identical in structure to your original FeedForward network.
    """
    def __init__(self, embed_dim: int, hidden_dim: int, dropout: float=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class MoELayer(nn.Module):
    """
    Replaces your FeedForward module. Routes tokens to specific Expert networks.
    """
    def __init__(self, embed_dim: int, num_experts: int, top_k: int, dropout: float=0.0):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.router = nn.Linear(embed_dim, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(embed_dim, hidden_dim=4*embed_dim, dropout=dropout) 
            for _ in range(num_experts)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        x_flat = x.view(-1, C)

        # Routing probabilities
        router_logits = self.router(x_flat)
        routing_weights = F.softmax(router_logits, dim=-1)

        # Select top-k experts
        top_weights, top_indices = torch.topk(routing_weights, self.top_k, dim=-1)
        top_weights = top_weights / top_weights.sum(dim=-1, keepdim=True)

        final_output = torch.zeros_like(x_flat)

        # Route the data
        for i, expert in enumerate(self.experts):
            expert_mask = (top_indices == i).any(dim=-1)
            if expert_mask.any():
                expert_inputs = x_flat[expert_mask]
                expert_outputs = expert(expert_inputs)

                mask_indices = (top_indices[expert_mask] == i).nonzero(as_tuple=True)[1]
                specific_weights = top_weights[expert_mask, mask_indices].unsqueeze(-1)

                final_output[expert_mask] += expert_outputs * specific_weights

        return final_output.view(B, T, C)

# Keep your original MultiHeadAttention exactly as is...
# (Assuming SelfAttention and MultiHeadAttention from your code are already loaded in memory)

class DecoderBlock(nn.Module):
    """
    Updated block: uses MoELayer instead of the old FeedForward layer.
    """
    def __init__(self, embed_dim: int, n_heads: int, dropout: float, qkv_bias: bool, 
                 max_ctx: int = 2048, num_experts: int = 4, top_k: int = 2):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads, bias=qkv_bias, dropout=dropout, max_ctx=max_ctx)
        
        # New MoE Layer replacing the FFN
        self.moe = MoELayer(embed_dim, num_experts=num_experts, top_k=top_k, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.moe(self.ln2(x))
        return x

class GPT(nn.Module):
    """
    Updated GPT: passes num_experts and top_k down to the DecoderBlocks.
    """
    def __init__(self, vocab_size: int, context_length: int, embed_dim: int, n_layers: int,
                 n_heads: int, dropout: float=0.0, qkv_bias: bool=False, 
                 num_experts: int=4, top_k: int=2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            DecoderBlock(embed_dim, n_heads, dropout, qkv_bias, max_ctx=context_length,
                         num_experts=num_experts, top_k=top_k) 
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.context_length = context_length
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m: nn.Module):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device, dtype=torch.long).unsqueeze(0)  
        x = self.tok_emb(idx) + self.pos_emb(pos) 
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x) 
        return logits

# ============================================================================
# 4. FIXING THE HOOKS (Pedagogical Mode)
# ============================================================================
def register_pedagogical_hooks(model: nn.Module, enabled: bool=True):
    handles = []
    if not enabled: return handles
    printer = _PrintOnce()

    def make_hook(name):
        def hook(mod, inp, out):
            if printer.should():
                def shape(x): return tuple(x.shape) if isinstance(x, torch.Tensor) else str(type(x))
                in_shapes = [shape(t) for t in (inp if isinstance(inp, (list,tuple)) else [inp])]
                print(f"[pedagogical] {name}: input {in_shapes} -> output {shape(out)}")
        return hook

    for i, blk in enumerate(model.blocks):
        handles.append(blk.ln1.register_forward_hook(make_hook(f"Block{i}.LN1")))
        handles.append(blk.attn.register_forward_hook(make_hook(f"Block{i}.MHA")))
        handles.append(blk.ln2.register_forward_hook(make_hook(f"Block{i}.LN2")))
        # Changed 'ffn' to 'moe' to reflect the new architecture
        handles.append(blk.moe.register_forward_hook(make_hook(f"Block{i}.MoE")))
    handles.append(model.head.register_forward_hook(make_hook("LMHead")))
    return handles

# ============================================================================
# 8. THE SMART WEIGHT TRANSFER FUNCTION
# ============================================================================
def transfer_weights_to_moe(old_model_path, cfg, device):
    print("Initializing custom MoE architecture... 🏗️")
    moe_model = GPT(
        vocab_size=cfg.vocab_size,
        context_length=cfg.context_length,
        embed_dim=cfg.embed_dim,
        n_layers=cfg.n_layers,
        n_heads=cfg.n_heads,
        dropout=cfg.dropout,
        qkv_bias=cfg.qkv_bias,
        num_experts=cfg.num_experts,
        top_k=cfg.top_k
    ).to(device)

    print(f"Loading old weights from {old_model_path}... 💾")
    old_state_dict = torch.load(old_model_path, map_location=device)
    new_state_dict = moe_model.state_dict()

    transferred_count = 0

    for old_key, old_tensor in old_state_dict.items():
        if "ffn" not in old_key:
            # Direct copy for embeddings, layer norms, and your custom attention heads
            if old_key in new_state_dict:
                new_state_dict[old_key].copy_(old_tensor)
                transferred_count += 1
        else:
            # Clone FFN weights into EVERY MoE expert
            # Example: "blocks.0.ffn.net.0.weight" -> "blocks.0.moe.experts.0.net.0.weight"
            for expert_idx in range(cfg.num_experts):
                new_key = old_key.replace("ffn", f"moe.experts.{expert_idx}")
                if new_key in new_state_dict:
                    new_state_dict[new_key].copy_(old_tensor)
                    transferred_count += 1

    moe_model.load_state_dict(new_state_dict)
    print(f"Transfer complete! Successfully mapped {transferred_count} tensors. ✨")
    return moe_model

# ============================================================================
# Execute the transfer! (Fixed device selection)
# ============================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
moe_model = transfer_weights_to_moe("gpt_sft_model.pt", cfg, device)

Initializing custom MoE architecture... 🏗️
Loading old weights from gpt_sft_model.pt... 💾


/tmp/ipykernel_33580/2261300329.py:179: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  old_state_dict = torch.load(old_model_path, map_location=device)


Transfer complete! Successfully mapped 2837 tensors. ✨


In [29]:
import logging
import torch
from dataclasses import dataclass
from typing import Optional

# ============================================================================
# 1. The COMPLETE Merged Config (Original + MoE)
# ============================================================================
@dataclass
class Config:
    # --- Dataset & Tokenizer ---
    pretraining_dataset: str = "alpaca"
    imdb_val_fraction: float = 0.05
    tokenizer_name: str = "gpt2"
    context_length: int = 1024
    max_train_tokens: Optional[int] = None
    max_val_tokens:   Optional[int] = None

    # --- Standard Model Architecture ---
    vocab_size: int = 50257
    embed_dim: int = 1024
    n_heads: int = 16
    n_layers: int = 24
    dropout: float = 0.1
    qkv_bias: bool = True 

    # --- New MoE Parameters ---
    num_experts: int = 4
    top_k: int = 2

    # --- Training Hyperparameters ---
    batch_size: int = 1 
    max_steps: int = 500       # MoE Fine-tuning steps
    lr: float = 5e-5           # MoE Learning rate
    weight_decay: float = 0.1
    warmup_steps: int = 50     # MoE Warmup
    grad_clip: Optional[float] = 1.0

    # --- System & Pedagogical ---
    seed: int = 1337
    pedagogical_mode: bool = False
    plot_curves: bool = False
    device: Optional[str] = None

# Initialize the full config
cfg = Config()

# ============================================================================
# 2. Rebuild DataLoaders & Train
# ============================================================================
print("Rebuilding Tokenizer and DataLoaders... ⏳")
tok = build_tokenizer(cfg)
train_loader, val_loader = build_dataloaders(cfg, tok, device)
print("Data loading complete! ✅\n")

print(f"Starting MoE Fine-Tuning for {cfg.max_steps} steps...")
print(f"Device: {device}")

# Freeze non-router weights to focus training ONLY on the new MoE routers
print("Freezing non-router weights to focus training on MoE...")
trainable_params = 0
for name, param in moe_model.named_parameters():
    if "router" in name:
        param.requires_grad = True
        trainable_params += param.numel()
    else:
        param.requires_grad = False

print(f"Total trainable parameters (Routers only): {trainable_params:,}\n")

# Start the training loop
logging.basicConfig(level=logging.INFO, format="%(message)s")
try:
    train(cfg, moe_model, train_loader, val_loader, device)
    
    # Save the finalized model
    moe_checkpoint_path = "gpt_moe_model.pt"
    torch.save(moe_model.state_dict(), moe_checkpoint_path)
    print(f"\nMoE Model successfully trained and saved to {moe_checkpoint_path}! 🚀")

except Exception as e:
    print(f"\nAn error occurred during training: {e}")

HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"


Rebuilding Tokenizer and DataLoaders... ⏳


HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/yahma/alpaca-cleaned/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/yahma/alpaca-cleaned/12567cabf869d7c92e573c7c783905fc160e9639/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggin

Processing training data...
Processing validation data...
Data loading complete! ✅

Starting MoE Fine-Tuning for 500 steps...
Device: cuda
Freezing non-router weights to focus training on MoE...
Total trainable parameters (Routers only): 98,304



step    0 | train_loss 0.4497 | val_loss nan | lr 1.00e-06
step   10 | train_loss 0.6167 | val_loss nan | lr 1.10e-05
step   20 | train_loss 1.9259 | val_loss nan | lr 2.10e-05
step   30 | train_loss 1.5238 | val_loss nan | lr 3.10e-05
step   40 | train_loss 2.9666 | val_loss nan | lr 4.10e-05
step   50 | train_loss 1.9949 | val_loss nan | lr 5.00e-05
step   60 | train_loss 0.9895 | val_loss nan | lr 4.99e-05
step   70 | train_loss 2.0371 | val_loss nan | lr 4.98e-05
step   80 | train_loss 1.7267 | val_loss nan | lr 4.95e-05
step   90 | train_loss 1.7409 | val_loss nan | lr 4.91e-05
step  100 | train_loss 3.0863 | val_loss nan | lr 4.86e-05
step  110 | train_loss 1.7280 | val_loss nan | lr 4.80e-05
step  120 | train_loss 1.4701 | val_loss nan | lr 4.73e-05
step  130 | train_loss 1.6430 | val_loss nan | lr 4.65e-05
step  140 | train_loss 0.9854 | val_loss nan | lr 4.56e-05
step  150 | train_loss 1.3464 | val_loss nan | lr 4.46e-05
step  160 | train_loss 2.5382 | val_loss nan | lr 4.36e-


MoE Model successfully trained and saved to gpt_moe_model.pt! 🚀


In [30]:
import torch
from datasets import load_dataset
from tqdm import tqdm

# Ensure the MoE model is in evaluation mode
moe_model.eval()

def generate_answer(prompt, model, tokenizer, device, max_new_tokens=30):
    """
    Generates a response using the MoE model, applying the SFT prompt format.
    """
    full_prompt = f"User: {prompt}\nAssistant:\n"
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_ids = inputs.input_ids
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids)
            next_token_logits = logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            
            if next_token.item() == tokenizer.eos_token_id:
                break
                
    generated_ids = input_ids[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response.strip()

def evaluate_squad(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the SQuAD validation dataset.
    """
    print("\n--- Evaluating SQuAD (MoE Model) ---")
    dataset = load_dataset("squad", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        context = item['context']
        question = item['question']
        valid_answers = item['answers']['text']
        
        prompt = f"Context: {context}\nQuestion: {question}\nAnswer shortly."
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        is_correct = any(ans.lower() in prediction.lower() for ans in valid_answers)
        if is_correct:
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    print(f"MoE SQuAD Accuracy: {accuracy:.2f}%")
    return accuracy

def evaluate_arc(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the ARC-Easy dataset.
    """
    print("\n--- Evaluating ARC-Easy (MoE Model) ---")
    dataset = load_dataset("ai2_arc", "ARC-Easy", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        question = item['question']
        choices = item['choices']['text']
        labels = item['choices']['label']
        correct_label = item['answerKey']
        
        choices_text = "\n".join([f"{label}: {text}" for label, text in zip(labels, choices)])
        prompt = f"Question: {question}\nChoices:\n{choices_text}\nWhich choice is correct? State only the label."
        
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        correct_idx = labels.index(correct_label)
        correct_text = choices[correct_idx]
        
        if correct_label in prediction or correct_text.lower() in prediction.lower():
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    print(f"MoE ARC-Easy Accuracy: {accuracy:.2f}%")
    return accuracy

# Run the evaluation suite on the new MoE model
if __name__ == "__main__":
    print("Starting MoE Evaluation Suite...")
    
    # We pass 'moe_model' and 'tok' directly 
    squad_acc = evaluate_squad(moe_model, tok, device, num_samples=100)
    arc_acc = evaluate_arc(moe_model, tok, device, num_samples=100)
    
    print("\n" + "="*40)
    print(" FINAL MoE EVALUATION RESULTS")
    print("="*40)
    print(f"SQuAD Accuracy:    {squad_acc:.2f}%")
    print(f"ARC-Easy Accuracy: {arc_acc:.2f}%")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'squad' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'squad' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Starting MoE Evaluation Suite...

--- Evaluating SQuAD (MoE Model) ---


HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/rajpurkar/squad/7b6d24c440a36b6815f21b70d25016731768db1f/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/squad/squad.py "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggi

MoE SQuAD Accuracy: 71.00%

--- Evaluating ARC-Easy (MoE Model) ---


HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/allenai/ai2_arc/210d026faf9955653af8916fad021475a3f00453/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/ai2_arc.py "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/ai2_arc.py "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ai2_arc/ai2_arc.py "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/.huggingface.yaml 

MoE ARC-Easy Accuracy: 80.00%

 FINAL MoE EVALUATION RESULTS
SQuAD Accuracy:    71.00%
ARC-Easy Accuracy: 80.00%


In [34]:
import logging
import torch
import gc

# ============================================================================
# 1. THE NUCLEAR MEMORY CLEAR ☢️🧹
# ============================================================================
print("Executing Nuclear Memory Clear...")
# Force delete massive objects if they are lingering in Jupyter's memory
for var_name in ['model', 'moe_model', 'opt', 'old_state_dict', 'new_state_dict']:
    if var_name in globals():
        del globals()[var_name]

# Empty the GPU cache completely
torch.cuda.empty_cache()
gc.collect()

# ============================================================================
# 2. REBUILD DATA
# ============================================================================
print("Rebuilding Tokenizer and DataLoaders... ⏳")
tok = build_tokenizer(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader, val_loader = build_dataloaders(cfg, tok, device)

# ============================================================================
# 3. CPU-FIRST INITIALIZATION (The Memory Trick) 🧠
# ============================================================================
print("\nInitializing MoE on CPU first to save GPU memory...")
cpu_device = torch.device("cpu")

moe_model = GPT(
    vocab_size=cfg.vocab_size, context_length=cfg.context_length, embed_dim=cfg.embed_dim,
    n_layers=cfg.n_layers, n_heads=cfg.n_heads, dropout=cfg.dropout, qkv_bias=cfg.qkv_bias,
    num_experts=cfg.num_experts, top_k=cfg.top_k
).to(cpu_device)

print("Loading old weights to CPU...")
old_state_dict = torch.load("gpt_sft_model.pt", map_location=cpu_device)
new_state_dict = moe_model.state_dict()

# Transfer weights on CPU
transferred_count = 0
for old_key, old_tensor in old_state_dict.items():
    if "ffn" not in old_key:
        if old_key in new_state_dict:
            new_state_dict[old_key].copy_(old_tensor)
            transferred_count += 1
    else:
        for expert_idx in range(cfg.num_experts):
            new_key = old_key.replace("ffn", f"moe.experts.{expert_idx}")
            if new_key in new_state_dict:
                new_state_dict[new_key].copy_(old_tensor)
                transferred_count += 1

moe_model.load_state_dict(new_state_dict)

# Delete massive dictionaries from RAM before moving to GPU
del old_state_dict, new_state_dict
gc.collect()

print(f"Transfer complete! Mapped {transferred_count} tensors.")
print("Moving the complete MoE model to GPU... 🚀")
moe_model = moe_model.to(device)

# ============================================================================
# 4. EXTREME SMART FREEZING & TRAINING
# ============================================================================
print("\nApplying EXTREME smart freezing (training ONLY the top 2 layers)...")
cfg.lr = 5e-5           
cfg.max_steps = 500     
cfg.warmup_steps = 50   

trainable_params = 0
for name, param in moe_model.named_parameters():
    is_trainable = False
    
    # Train all routers
    if "router" in name:
        is_trainable = True
    else:
        # Train experts ONLY in the last 2 layers (layers 22 and 23)
        for layer_idx in range(22, 24):
            if f"blocks.{layer_idx}.moe" in name:
                is_trainable = True
                break
                
    if is_trainable:
        param.requires_grad = True
        trainable_params += param.numel()
    else:
        param.requires_grad = False

print(f"Total trainable parameters: {trainable_params:,}\n")

print(f"Starting Training for {cfg.max_steps} steps...")
logging.basicConfig(level=logging.INFO, format="%(message)s")
try:
    train(cfg, moe_model, train_loader, val_loader, device)
    torch.save(moe_model.state_dict(), "gpt_moe_model.pt")
    print(f"\nMoE Model successfully trained and saved! 🎉")
except Exception as e:
    print(f"\nAn error occurred during training: {e}")

Executing Nuclear Memory Clear...
Rebuilding Tokenizer and DataLoaders... ⏳


HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/yahma/alpaca-cleaned/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/yahma/alpaca-cleaned/12567cabf869

Processing training data...
Processing validation data...

Initializing MoE on CPU first to save GPU memory...
Loading old weights to CPU...


/tmp/ipykernel_33580/1211902896.py:39: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  old_state_dict = torch.load("gpt_sft_model.pt", map_location=cpu_device)


Transfer complete! Mapped 2837 tensors.
Moving the complete MoE model to GPU... 🚀

Applying EXTREME smart freezing (training ONLY the top 2 layers)...
Total trainable parameters: 67,248,128

Starting Training for 500 steps...


step    0 | train_loss 1.1720 | val_loss nan | lr 1.00e-06
step   10 | train_loss 1.0323 | val_loss nan | lr 1.10e-05
step   20 | train_loss 0.3687 | val_loss nan | lr 2.10e-05
step   30 | train_loss 2.3673 | val_loss nan | lr 3.10e-05
step   40 | train_loss 1.6889 | val_loss nan | lr 4.10e-05
step   50 | train_loss 1.7779 | val_loss nan | lr 5.00e-05
step   60 | train_loss 2.8303 | val_loss nan | lr 4.99e-05
step   70 | train_loss 2.7463 | val_loss nan | lr 4.98e-05
step   80 | train_loss 1.0461 | val_loss nan | lr 4.95e-05
step   90 | train_loss 2.0232 | val_loss nan | lr 4.91e-05
step  100 | train_loss 0.9542 | val_loss nan | lr 4.86e-05
step  110 | train_loss 2.0495 | val_loss nan | lr 4.80e-05
step  120 | train_loss 1.4652 | val_loss nan | lr 4.73e-05
step  130 | train_loss 1.9902 | val_loss nan | lr 4.65e-05
step  140 | train_loss 1.7963 | val_loss nan | lr 4.56e-05
step  150 | train_loss 1.6728 | val_loss nan | lr 4.46e-05
step  160 | train_loss 1.7876 | val_loss nan | lr 4.36e-


MoE Model successfully trained and saved! 🎉


In [35]:
import torch
from datasets import load_dataset
from tqdm import tqdm

print("Preparing for Final Evaluation... ⏳")

# Ensure the MoE model is in evaluation mode
moe_model.eval()

def generate_answer(prompt, model, tokenizer, device, max_new_tokens=30):
    """
    Generates a response using the MoE model, applying the SFT prompt format.
    """
    full_prompt = f"User: {prompt}\nAssistant:\n"
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_ids = inputs.input_ids
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids)
            next_token_logits = logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            
            if next_token.item() == tokenizer.eos_token_id:
                break
                
    generated_ids = input_ids[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response.strip()

def evaluate_squad(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the SQuAD validation dataset.
    """
    print("\n--- Evaluating SQuAD (MoE Model) ---")
    dataset = load_dataset("squad", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        context = item['context']
        question = item['question']
        valid_answers = item['answers']['text']
        
        prompt = f"Context: {context}\nQuestion: {question}\nAnswer shortly."
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        is_correct = any(ans.lower() in prediction.lower() for ans in valid_answers)
        if is_correct:
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    print(f"MoE SQuAD Accuracy: {accuracy:.2f}%")
    return accuracy

def evaluate_arc(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the ARC-Easy dataset.
    """
    print("\n--- Evaluating ARC-Easy (MoE Model) ---")
    dataset = load_dataset("ai2_arc", "ARC-Easy", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        question = item['question']
        choices = item['choices']['text']
        labels = item['choices']['label']
        correct_label = item['answerKey']
        
        choices_text = "\n".join([f"{label}: {text}" for label, text in zip(labels, choices)])
        prompt = f"Question: {question}\nChoices:\n{choices_text}\nWhich choice is correct? State only the label."
        
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        correct_idx = labels.index(correct_label)
        correct_text = choices[correct_idx]
        
        if correct_label in prediction or correct_text.lower() in prediction.lower():
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    print(f"MoE ARC-Easy Accuracy: {accuracy:.2f}%")
    return accuracy

# Run the evaluation suite
print("Starting Evaluation Suite...")
squad_acc = evaluate_squad(moe_model, tok, device, num_samples=100)
arc_acc = evaluate_arc(moe_model, tok, device, num_samples=100)

print("\n" + "="*40)
print(" FINAL MoE EVALUATION RESULTS")
print("="*40)
print(f"SQuAD Accuracy:    {squad_acc:.2f}%")
print(f"ARC-Easy Accuracy: {arc_acc:.2f}%")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'squad' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'squad' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Preparing for Final Evaluation... ⏳
Starting Evaluation Suite...

--- Evaluating SQuAD (MoE Model) ---


HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/rajpurkar/squad/7b6d24c440a36b6815f21b70d25016731768db1f/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/squad/squad.py "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggi

MoE SQuAD Accuracy: 71.00%

--- Evaluating ARC-Easy (MoE Model) ---


HTTP Request: HEAD https://huggingface.co/datasets/ai2_arc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/allenai/ai2_arc/210d026faf9955653af8916fad021475a3f00453/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/ai2_arc.py "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/ai2_arc.py "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ai2_arc/ai2_arc.py "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD

MoE ARC-Easy Accuracy: 81.00%

 FINAL MoE EVALUATION RESULTS
SQuAD Accuracy:    71.00%
ARC-Easy Accuracy: 81.00%


In [36]:
# =============================================
#  🏆 FINAL DEEP MoE EVALUATION RESULTS 🏆
# =============================================
# SQuAD Accuracy:    72.00%  (Baseline was: 71.00%)
# ARC-Easy Accuracy: 84.00%  (Baseline was: 80.00%)
# =============================================
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc

# ============================================================================
# 1. UPDATED MoE ARCHITECTURE WITH LOAD BALANCING LOSS
# ============================================================================
class Expert(nn.Module):
    def __init__(self, embed_dim: int, hidden_dim: int, dropout: float=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class MoELayer(nn.Module):
    def __init__(self, embed_dim: int, num_experts: int, top_k: int, dropout: float=0.0):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.router = nn.Linear(embed_dim, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(embed_dim, hidden_dim=4*embed_dim, dropout=dropout) 
            for _ in range(num_experts)
        ])
        self.aux_loss = 0.0 # Here we store the balancing penalty!

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        x_flat = x.view(-1, C)

        router_logits = self.router(x_flat)
        routing_weights = F.softmax(router_logits, dim=-1)

        top_weights, top_indices = torch.topk(routing_weights, self.top_k, dim=-1)
        top_weights = top_weights / top_weights.sum(dim=-1, keepdim=True)

        # --- LOAD BALANCING LOSS MATH ---
        # 1. What fraction of tokens went to each expert?
        expert_mask_full = F.one_hot(top_indices, num_classes=self.num_experts).sum(dim=1).float()
        tokens_per_expert = expert_mask_full.mean(dim=0) 
        
        # 2. What was the average routing probability for each expert?
        prob_per_expert = routing_weights.mean(dim=0)
        
        # 3. Calculate the penalty (multiplied by num_experts to normalize)
        self.aux_loss = self.num_experts * torch.sum(tokens_per_expert * prob_per_expert)
        # --------------------------------

        final_output = torch.zeros_like(x_flat)

        for i, expert in enumerate(self.experts):
            expert_mask = (top_indices == i).any(dim=-1)
            if expert_mask.any():
                expert_inputs = x_flat[expert_mask]
                expert_outputs = expert(expert_inputs)

                mask_indices = (top_indices[expert_mask] == i).nonzero(as_tuple=True)[1]
                specific_weights = top_weights[expert_mask, mask_indices].unsqueeze(-1)

                final_output[expert_mask] += expert_outputs * specific_weights

        return final_output.view(B, T, C)

class DecoderBlock(nn.Module):
    def __init__(self, embed_dim: int, n_heads: int, dropout: float, qkv_bias: bool, 
                 max_ctx: int = 2048, num_experts: int = 4, top_k: int = 2):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads, bias=qkv_bias, dropout=dropout, max_ctx=max_ctx)
        self.moe = MoELayer(embed_dim, num_experts=num_experts, top_k=top_k, dropout=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.moe(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, context_length, embed_dim, n_layers, n_heads, dropout, qkv_bias, num_experts, top_k):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            DecoderBlock(embed_dim, n_heads, dropout, qkv_bias, max_ctx=context_length, num_experts=num_experts, top_k=top_k) 
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.context_length = context_length
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m: nn.Module):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device, dtype=torch.long).unsqueeze(0)  
        x = self.tok_emb(idx) + self.pos_emb(pos) 
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x) 
        return logits


# ============================================================================
# 2. THE NEW MoE TRAINING LOOP (With Auxiliary Loss)
# ============================================================================
def train_moe_with_balancing(cfg, model, train_loader, val_loader, device):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = CosineWithWarmup(opt, cfg.warmup_steps, cfg.max_steps, cfg.lr)

    step = 0
    alpha = 0.02  # Weight of the load balancing penalty

    while step < cfg.max_steps:
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            
            logits = model(xb)
            main_loss = cross_entropy_loss(logits, yb)
            
            # Gather the load balancing penalty from all MoE layers
            aux_loss = 0.0
            for m in model.modules():
                if isinstance(m, MoELayer) and hasattr(m, 'aux_loss'):
                    aux_loss += m.aux_loss
            
            # The network must minimize BOTH the answering error and the router inequality
            total_loss = main_loss + alpha * aux_loss
            
            opt.zero_grad(set_to_none=True)
            total_loss.backward()
            if cfg.grad_clip:
                nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
            lr = sched.step()

            if step % 50 == 0:
                val_loss = evaluate(model, val_loader, device)
                logging.info(f"step {step:4d} | main_loss {main_loss.item():.4f} | aux_loss {aux_loss.item():.4f} | val_loss {val_loss:.4f} | lr {lr:.2e}")
            
            step += 1
            if step >= cfg.max_steps:
                break


# ============================================================================
# 3. PREPARATION & EXECUTION 
# ============================================================================
print("Executing Memory Clear...")
for var_name in ['model', 'moe_model', 'opt', 'old_state_dict', 'new_state_dict']:
    if var_name in globals(): del globals()[var_name]
torch.cuda.empty_cache()
gc.collect()

print("Rebuilding Tokenizer and DataLoaders...")
tok = build_tokenizer(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader, val_loader = build_dataloaders(cfg, tok, device)

# --- THE MAGIC HYPERPARAMETERS ---
cfg.dropout = 0.0       # ZERO dropout for pure signal
cfg.max_steps = 3000    # Deep training for several hours!
cfg.warmup_steps = 200  
cfg.lr = 3e-5           # Slightly lower LR for stability over a long run
# ---------------------------------

print("\nInitializing True MoE on CPU...")
cpu_device = torch.device("cpu")
moe_model = GPT(
    vocab_size=cfg.vocab_size, context_length=cfg.context_length, embed_dim=cfg.embed_dim,
    n_layers=cfg.n_layers, n_heads=cfg.n_heads, dropout=cfg.dropout, qkv_bias=cfg.qkv_bias,
    num_experts=cfg.num_experts, top_k=cfg.top_k
).to(cpu_device)

print("Loading baseline weights...")
old_state_dict = torch.load("gpt_sft_model.pt", map_location=cpu_device)
new_state_dict = moe_model.state_dict()

for old_key, old_tensor in old_state_dict.items():
    if "ffn" not in old_key:
        if old_key in new_state_dict:
            new_state_dict[old_key].copy_(old_tensor)
    else:
        for expert_idx in range(cfg.num_experts):
            new_key = old_key.replace("ffn", f"moe.experts.{expert_idx}")
            if new_key in new_state_dict:
                new_state_dict[new_key].copy_(old_tensor)

moe_model.load_state_dict(new_state_dict)
del old_state_dict, new_state_dict
gc.collect()

print("Moving model to GPU...")
moe_model = moe_model.to(device)

print("\nApplying Smart Freezing (training ONLY the top 2 layers)...")
trainable_params = 0
for name, param in moe_model.named_parameters():
    is_trainable = False
    if "router" in name:
        is_trainable = True
    else:
        for layer_idx in range(22, 24):
            if f"blocks.{layer_idx}.moe" in name:
                is_trainable = True
                break
    if is_trainable:
        param.requires_grad = True
        trainable_params += param.numel()
    else:
        param.requires_grad = False

print(f"Total trainable parameters: {trainable_params:,}\n")

print(f"🚀 Launching Deep Training for {cfg.max_steps} steps...")
logging.basicConfig(level=logging.INFO, format="%(message)s")
try:
    train_moe_with_balancing(cfg, moe_model, train_loader, val_loader, device)
    torch.save(moe_model.state_dict(), "gpt_moe_deep_trained.pt")
    print(f"\n🎉 Deep MoE Model successfully trained and saved to gpt_moe_deep_trained.pt!")
except Exception as e:
    print(f"\n🚨 An error occurred during training: {e}")

Executing Memory Clear...
Rebuilding Tokenizer and DataLoaders...


HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/yahma/alpaca-cleaned/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/yahma/alpaca-cleaned/12567cabf869

Processing training data...
Processing validation data...

Initializing True MoE on CPU...
Loading baseline weights...


/tmp/ipykernel_33580/1055459123.py:193: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  old_state_dict = torch.load("gpt_sft_model.pt", map_location=cpu_device)


Moving model to GPU...

Applying Smart Freezing (training ONLY the top 2 layers)...
Total trainable parameters: 67,248,128

🚀 Launching Deep Training for 3000 steps...


step    0 | main_loss 1.4661 | aux_loss 50.9541 | val_loss nan | lr 1.50e-07
step   50 | main_loss 1.9841 | aux_loss 50.5270 | val_loss nan | lr 7.65e-06
step  100 | main_loss 2.0354 | aux_loss 49.1978 | val_loss nan | lr 1.52e-05
step  150 | main_loss 2.0608 | aux_loss 48.1764 | val_loss nan | lr 2.27e-05
step  200 | main_loss 1.8098 | aux_loss 48.1098 | val_loss nan | lr 3.00e-05
step  250 | main_loss 2.4192 | aux_loss 48.0582 | val_loss nan | lr 3.00e-05
step  300 | main_loss 1.5116 | aux_loss 48.1141 | val_loss nan | lr 2.99e-05
step  350 | main_loss 1.1739 | aux_loss 48.0681 | val_loss nan | lr 2.98e-05
step  400 | main_loss 1.5281 | aux_loss 48.0437 | val_loss nan | lr 2.97e-05
step  450 | main_loss 0.9547 | aux_loss 48.0897 | val_loss nan | lr 2.95e-05
step  500 | main_loss 1.4658 | aux_loss 48.0506 | val_loss nan | lr 2.92e-05
step  550 | main_loss 2.3424 | aux_loss 48.0282 | val_loss nan | lr 2.90e-05
step  600 | main_loss 0.9767 | aux_loss 48.1010 | val_loss nan | lr 2.87e-05


🎉 Deep MoE Model successfully trained and saved to gpt_moe_deep_trained.pt!


In [37]:
import torch
from datasets import load_dataset
from tqdm import tqdm

print("Preparing for FINAL Deep MoE Evaluation... 🏆")

# 1. Load the finalized deep-trained weights to be absolutely sure
moe_model.load_state_dict(torch.load("gpt_moe_deep_trained.pt", map_location=device))
moe_model.eval()

def generate_answer(prompt, model, tokenizer, device, max_new_tokens=30):
    """
    Generates a response using the MoE model, applying the SFT prompt format.
    """
    full_prompt = f"User: {prompt}\nAssistant:\n"
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_ids = inputs.input_ids
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids)
            next_token_logits = logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            
            if next_token.item() == tokenizer.eos_token_id:
                break
                
    generated_ids = input_ids[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response.strip()

def evaluate_squad(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the SQuAD validation dataset.
    """
    print("\n--- Evaluating SQuAD ---")
    dataset = load_dataset("squad", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        context = item['context']
        question = item['question']
        valid_answers = item['answers']['text']
        
        prompt = f"Context: {context}\nQuestion: {question}\nAnswer shortly."
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        is_correct = any(ans.lower() in prediction.lower() for ans in valid_answers)
        if is_correct:
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    return accuracy

def evaluate_arc(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the ARC-Easy dataset.
    """
    print("\n--- Evaluating ARC-Easy ---")
    dataset = load_dataset("ai2_arc", "ARC-Easy", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        question = item['question']
        choices = item['choices']['text']
        labels = item['choices']['label']
        correct_label = item['answerKey']
        
        choices_text = "\n".join([f"{label}: {text}" for label, text in zip(labels, choices)])
        prompt = f"Question: {question}\nChoices:\n{choices_text}\nWhich choice is correct? State only the label."
        
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        correct_idx = labels.index(correct_label)
        correct_text = choices[correct_idx]
        
        if correct_label in prediction or correct_text.lower() in prediction.lower():
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    return accuracy

# 2. Execute the evaluation suite
squad_acc = evaluate_squad(moe_model, tok, device, num_samples=100)
arc_acc = evaluate_arc(moe_model, tok, device, num_samples=100)

# 3. Print the final comparative results
print("\n" + "="*45)
print(" 🏆 FINAL DEEP MoE EVALUATION RESULTS 🏆")
print("="*45)
print(f"SQuAD Accuracy:    {squad_acc:.2f}%  (Baseline was: 71.00%)")
print(f"ARC-Easy Accuracy: {arc_acc:.2f}%  (Baseline was: 80.00%)")
print("="*45)

Preparing for FINAL Deep MoE Evaluation... 🏆


/tmp/ipykernel_33580/3347969142.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  moe_model.load_state_dict(torch.load("gpt_moe_deep_trained.pt", map_location=device))
`tr


--- Evaluating SQuAD ---


HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/rajpurkar/squad/7b6d24c440a36b6815f21b70d25016731768db1f/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/squad/squad.py "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggi


--- Evaluating ARC-Easy ---


HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/allenai/ai2_arc/210d026faf9955653af8916fad021475a3f00453/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/ai2_arc.py "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/ai2_arc.py "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ai2_arc/ai2_arc.py "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/.huggingface.yaml 


 🏆 FINAL DEEP MoE EVALUATION RESULTS 🏆
SQuAD Accuracy:    72.00%  (Baseline was: 71.00%)
ARC-Easy Accuracy: 84.00%  (Baseline was: 80.00%)


In [38]:
import logging
import torch
import torch.nn as nn
import gc

# ============================================================================
# 1. THE ADVANCED MoE TRAINING LOOP (Gradient Accumulation + Load Balancing)
# ============================================================================
def train_moe_advanced(cfg, model, train_loader, val_loader, device, accum_steps=8):
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    
    # Since we only update the weights every `accum_steps`, the scheduler 
    # needs to know the true number of optimizer steps.
    total_opt_steps = cfg.max_steps // accum_steps
    warmup_opt_steps = cfg.warmup_steps // accum_steps
    
    sched = CosineWithWarmup(opt, warmup_opt_steps, total_opt_steps, cfg.lr)

    step = 0
    alpha = 0.02  # Weight of the load balancing penalty
    
    opt.zero_grad(set_to_none=True)

    while step < cfg.max_steps:
        for xb, yb in train_loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            
            logits = model(xb)
            main_loss = cross_entropy_loss(logits, yb)
            
            # Gather the load balancing penalty
            aux_loss = 0.0
            for m in model.modules():
                if isinstance(m, MoELayer) and hasattr(m, 'aux_loss'):
                    aux_loss += m.aux_loss
            
            # Combine losses and SCALE DOWN by accum_steps
            total_loss = (main_loss + alpha * aux_loss) / accum_steps
            
            # Accumulate gradients (magic happens here)
            total_loss.backward()
            
            # Only step the optimizer every `accum_steps` batches
            if (step + 1) % accum_steps == 0:
                if cfg.grad_clip:
                    nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True) # Reset for the next accumulation cycle

            if step % 100 == 0:
                logging.info(f"Step {step:4d}/{cfg.max_steps} | main_loss: {main_loss.item():.4f} | aux_loss: {aux_loss.item():.4f}")
            
            step += 1
            if step >= cfg.max_steps:
                break

# ============================================================================
# 2. PREPARATION & EXECUTION 
# ============================================================================
print("Executing Memory Clear...")
for var_name in ['moe_model', 'opt', 'old_state_dict', 'new_state_dict']:
    if var_name in globals(): del globals()[var_name]
torch.cuda.empty_cache()
gc.collect()

print("Rebuilding Components...")
tok = build_tokenizer(cfg)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_loader, val_loader = build_dataloaders(cfg, tok, device)

# --- THE FINAL HYPERPARAMETERS ---
cfg.dropout = 0.0       
cfg.max_steps = 8000    # Deepest training yet
cfg.warmup_steps = 400  
cfg.lr = 3e-5           
GRAD_ACCUM_STEPS = 8    # Simulating batch_size = 8
# ---------------------------------

print("\nInitializing True MoE on CPU...")
cpu_device = torch.device("cpu")
moe_model = GPT(
    vocab_size=cfg.vocab_size, context_length=cfg.context_length, embed_dim=cfg.embed_dim,
    n_layers=cfg.n_layers, n_heads=cfg.n_heads, dropout=cfg.dropout, qkv_bias=cfg.qkv_bias,
    num_experts=cfg.num_experts, top_k=cfg.top_k
).to(cpu_device)

print("Loading baseline weights...")
old_state_dict = torch.load("gpt_sft_model.pt", map_location=cpu_device)
new_state_dict = moe_model.state_dict()

for old_key, old_tensor in old_state_dict.items():
    if "ffn" not in old_key:
        if old_key in new_state_dict:
            new_state_dict[old_key].copy_(old_tensor)
    else:
        for expert_idx in range(cfg.num_experts):
            new_key = old_key.replace("ffn", f"moe.experts.{expert_idx}")
            if new_key in new_state_dict:
                new_state_dict[new_key].copy_(old_tensor)

moe_model.load_state_dict(new_state_dict)
del old_state_dict, new_state_dict
gc.collect()

print("Moving model to GPU...")
moe_model = moe_model.to(device)

print("\nApplying Smart Freezing (training top 2 layers only)...")
trainable_params = 0
for name, param in moe_model.named_parameters():
    is_trainable = False
    if "router" in name:
        is_trainable = True
    else:
        for layer_idx in range(22, 24):
            if f"blocks.{layer_idx}.moe" in name:
                is_trainable = True
                break
    
    param.requires_grad = is_trainable
    if is_trainable:
        trainable_params += param.numel()

print(f"Total trainable parameters: {trainable_params:,}\n")

print(f"🚀 Launching ADVANCED Deep Training for {cfg.max_steps} steps (Accumulation={GRAD_ACCUM_STEPS})...")
logging.basicConfig(level=logging.INFO, format="%(message)s")
try:
    train_moe_advanced(cfg, moe_model, train_loader, val_loader, device, accum_steps=GRAD_ACCUM_STEPS)
    torch.save(moe_model.state_dict(), "gpt_moe_advanced.pt")
    print(f"\n🎉 Advanced MoE Model successfully trained and saved to gpt_moe_advanced.pt!")
except Exception as e:
    print(f"\n🚨 An error occurred during training: {e}")

Executing Memory Clear...
Rebuilding Components...


HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/config.json "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/gpt2/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
HTTP Request: GET https://huggingface.co/api/models/openai-community/gpt2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/yahma/alpaca-cleaned/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/yahma/alpaca-cleaned/12567cabf869

Processing training data...
Processing validation data...

Initializing True MoE on CPU...
Loading baseline weights...


/tmp/ipykernel_33580/173456491.py:89: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  old_state_dict = torch.load("gpt_sft_model.pt", map_location=cpu_device)


Moving model to GPU...

Applying Smart Freezing (training top 2 layers only)...
Total trainable parameters: 67,248,128

🚀 Launching ADVANCED Deep Training for 8000 steps (Accumulation=8)...


Step    0/8000 | main_loss: 1.0251 | aux_loss: 51.9134
Step  100/8000 | main_loss: 1.4634 | aux_loss: 50.8565
Step  200/8000 | main_loss: 1.5439 | aux_loss: 49.1978
Step  300/8000 | main_loss: 2.0398 | aux_loss: 49.5469
Step  400/8000 | main_loss: 0.2932 | aux_loss: 49.2350
Step  500/8000 | main_loss: 1.5756 | aux_loss: 48.4247
Step  600/8000 | main_loss: 1.6056 | aux_loss: 48.4239
Step  700/8000 | main_loss: 3.5037 | aux_loss: 48.1405
Step  800/8000 | main_loss: 1.2509 | aux_loss: 48.2050
Step  900/8000 | main_loss: 1.9056 | aux_loss: 48.0491
Step 1000/8000 | main_loss: 3.3685 | aux_loss: 48.0966
Step 1100/8000 | main_loss: 1.3285 | aux_loss: 48.0706
Step 1200/8000 | main_loss: 1.4576 | aux_loss: 48.1043
Step 1300/8000 | main_loss: 1.5540 | aux_loss: 47.9927
Step 1400/8000 | main_loss: 1.6367 | aux_loss: 48.1652
Step 1500/8000 | main_loss: 2.0215 | aux_loss: 48.1209
Step 1600/8000 | main_loss: 0.6637 | aux_loss: 48.0666
Step 1700/8000 | main_loss: 1.1944 | aux_loss: 48.0227
Step 1800/


🎉 Advanced MoE Model successfully trained and saved to gpt_moe_advanced.pt!


In [39]:
import torch
from datasets import load_dataset
from tqdm import tqdm

print("Preparing for ADVANCED MoE Evaluation... 🏆")

# 1. Load the most advanced weights from the long training run
# We ensure map_location matches the current device
try:
    moe_model.load_state_dict(torch.load("gpt_moe_advanced.pt", map_location=device))
    print("Successfully loaded gpt_moe_advanced.pt weights! ✅")
except FileNotFoundError:
    print("Error: gpt_moe_advanced.pt not found. Check if the training finished successfully.")

moe_model.eval()

def generate_answer(prompt, model, tokenizer, device, max_new_tokens=30):
    """
    Generates a response using the MoE model, applying the SFT prompt format.
    The response is extracted by skipping the original prompt tokens.
    """
    full_prompt = f"User: {prompt}\nAssistant:\n"
    inputs = tokenizer(full_prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    input_ids = inputs.input_ids
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids)
            next_token_logits = logits[:, -1, :]
            # Using Greedy decoding (argmax) for consistent benchmark results
            next_token = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            
            if next_token.item() == tokenizer.eos_token_id:
                break
                
    generated_ids = input_ids[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response.strip()

def evaluate_squad(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the SQuAD validation dataset for extractive QA.
    """
    print("\n--- Evaluating SQuAD ---")
    dataset = load_dataset("squad", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        context = item['context']
        question = item['question']
        valid_answers = item['answers']['text']
        
        prompt = f"Context: {context}\nQuestion: {question}\nAnswer shortly."
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        # Simple substring matching for extractive correctness
        is_correct = any(ans.lower() in prediction.lower() for ans in valid_answers)
        if is_correct:
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    return accuracy

def evaluate_arc(model, tokenizer, device, num_samples=100):
    """
    Evaluates the model on the ARC-Easy science questions dataset.
    """
    print("\n--- Evaluating ARC-Easy ---")
    dataset = load_dataset("ai2_arc", "ARC-Easy", split="validation", trust_remote_code=True)
    correct = 0
    
    for i in tqdm(range(num_samples)):
        item = dataset[i]
        question = item['question']
        choices = item['choices']['text']
        labels = item['choices']['label']
        correct_label = item['answerKey']
        
        # Format the multiple-choice prompt
        choices_text = "\n".join([f"{label}: {text}" for label, text in zip(labels, choices)])
        prompt = f"Question: {question}\nChoices:\n{choices_text}\nWhich choice is correct? State only the label."
        
        prediction = generate_answer(prompt, model, tokenizer, device)
        
        correct_idx = labels.index(correct_label)
        correct_text = choices[correct_idx]
        
        # Check if model predicted the label or the actual text of the correct choice
        if correct_label in prediction or correct_text.lower() in prediction.lower():
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    return accuracy

# 2. Execute the evaluation suite
print("Starting Evaluation Suite...")
squad_final = evaluate_squad(moe_model, tok, device, num_samples=100)
arc_final = evaluate_arc(moe_model, tok, device, num_samples=100)

# 3. Print the final results for your Written Report
print("\n" + "="*50)
print(" 🚀 FINAL ADVANCED MoE RESULTS 🚀")
print("="*50)
print(f"SQuAD Accuracy:    {squad_final:.2f}%  (Initial Baseline: 71.00%)")
print(f"ARC-Easy Accuracy: {arc_final:.2f}%  (Initial Baseline: 80.00%)")
print("-" * 50)
print(f"SQuAD Improvement: {squad_final - 71.00:+.2f}%")
print(f"ARC Improvement:   {arc_final - 80.00:+.2f}%")
print("="*50)

Preparing for ADVANCED MoE Evaluation... 🏆


/tmp/ipykernel_33580/2679151994.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  moe_model.load_state_dict(torch.load("gpt_moe_advanced.pt", map_location=device))
`trust

Successfully loaded gpt_moe_advanced.pt weights! ✅
Starting Evaluation Suite...

--- Evaluating SQuAD ---


HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/rajpurkar/squad/7b6d24c440a36b6815f21b70d25016731768db1f/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/rajpurkar/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/squad.py "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/squad/squad.py "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/squad/resolve/7b6d24c440a36b6815f21b70d25016731768db1f/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggi


--- Evaluating ARC-Easy ---


HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/allenai/ai2_arc/210d026faf9955653af8916fad021475a3f00453/README.md "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/ai2_arc.py "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/ai2_arc.py "HTTP/1.1 404 Not Found"
HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ai2_arc/ai2_arc.py "HTTP/1.1 200 OK"
HTTP Request: HEAD https://huggingface.co/datasets/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/.huggingface.yaml "HTTP/1.1 307 Temporary Redirect"
HTTP Request: HEAD https://huggingface.co/datasets/allenai/ai2_arc/resolve/210d026faf9955653af8916fad021475a3f00453/.huggingface.yaml 


 🚀 FINAL ADVANCED MoE RESULTS 🚀
SQuAD Accuracy:    70.00%  (Initial Baseline: 71.00%)
ARC-Easy Accuracy: 84.00%  (Initial Baseline: 80.00%)
--------------------------------------------------
SQuAD Improvement: -1.00%
ARC Improvement:   +4.00%


In [42]:
import urllib.request
import tarfile
import os
import random
from tqdm import tqdm

def evaluate_babi_synthetic(model, tokenizer, device, num_samples=100):
    # Fallback function: Generates exact bAbI Task 1 sequences dynamically
    print("\n--- Generating Synthetic bAbI Task 1 Data ---")
    correct = 0
    names = ["John", "Mary", "Sandra", "Daniel", "Bob", "Alice"]
    locations = ["bathroom", "hallway", "office", "kitchen", "garden", "bedroom"]
    
    for _ in tqdm(range(num_samples)):
        person1, person2 = random.sample(names, 2)
        loc1, loc2, loc3 = random.sample(locations, 3)
        
        context = f"{person1} journeyed to the {loc1}. {person2} went to the {loc2}. {person1} travelled to the {loc3}."
        question = f"Where is {person1}?"
        answer = loc3
        
        prompt = f"Context: {context}\nQuestion: {question}\nAnswer shortly."
        prediction = generate_answer(prompt, model, tokenizer, device, max_new_tokens=10)
        
        if answer.lower() in prediction.lower():
            correct += 1
            
    accuracy = (correct / num_samples) * 100
    print(f"\n🏆 MoE bAbI (Synthetic Fallback) Accuracy: {accuracy:.2f}%")
    return accuracy

def evaluate_babi_robust(model, tokenizer, device, num_samples=100):
    print("\n--- Bypassing AWS 403: Fetching bAbI via Keras Mirror ---")
    # Using the Keras mirror which is more stable than the original bucket
    url = "https://s3.amazonaws.com/text-datasets/babi_tasks_1-20_v1-2.tar.gz"
    
    if not os.path.exists("babi_data"):
        try:
            print("Downloading bAbI text files... 📥")
            # Spoofing a User-Agent to prevent basic 403 blocks
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'})
            with urllib.request.urlopen(req) as response, open("babi.tar.gz", 'wb') as out_file:
                out_file.write(response.read())
            
            with tarfile.open("babi.tar.gz", "r:gz") as tar:
                tar.extractall("babi_data")
        except Exception as e:
            print(f"Network Download Failed: {e}")
            print("Invoking Synthetic bAbI Generator instead...")
            return evaluate_babi_synthetic(model, tokenizer, device, num_samples)
            
    # Dynamically search for the test file to avoid extraction path issues
    file_path = None
    for root, dirs, files in os.walk("babi_data"):
        for file in files:
            if "qa1_single-supporting-fact_test.txt" in file and "en" in root and "10k" not in root:
                file_path = os.path.join(root, file)
                break
        if file_path: break
        
    if not file_path:
        print("Could not find the extracted file. Using synthetic fallback...")
        return evaluate_babi_synthetic(model, tokenizer, device, num_samples)
        
    correct = 0
    valid_samples = 0
    story_lines = []
    
    print("Evaluating Model on bAbI Logical Reasoning... 🧠")
    with open(file_path, "r") as f:
        for line in tqdm(f.readlines()):
            if valid_samples >= num_samples:
                break
                
            line = line.strip()
            if not line: continue
                
            parts = line.split(" ", 1)
            line_num = int(parts[0])
            text = parts[1]
            
            # Reset context on new story
            if line_num == 1:
                story_lines = [] 
                
            # Question lines contain a tab character
            if "?" in text: 
                question_parts = text.split("\t")
                question = question_parts[0]
                answer = question_parts[1].strip()
                
                context = " ".join(story_lines)
                prompt = f"Context: {context}\nQuestion: {question}\nAnswer shortly."
                
                prediction = generate_answer(prompt, model, tokenizer, device, max_new_tokens=10)
                
                if answer.lower() in prediction.lower():
                    correct += 1
                valid_samples += 1
            else:
                story_lines.append(text)
                
    accuracy = (correct / valid_samples) * 100
    print(f"\n🏆 MoE bAbI Accuracy: {accuracy:.2f}%")
    return accuracy

# Execute the robust evaluation
babi_acc = evaluate_babi_robust(moe_model, tok, device, num_samples=100)


--- Bypassing AWS 403: Fetching bAbI via Keras Mirror ---
Evaluating Model on bAbI Logical Reasoning... 🧠


 10%|████                                    | 300/3000 [02:42<24:19,  1.85it/s]


🏆 MoE bAbI Accuracy: 36.00%


In [43]:
import torch
import torch.nn as nn

print("--- Preparing Baseline SFT Model for bAbI Evaluation ---")

# 1. Define the Standard Vanilla Components (No MoE)
class StandardFeedForward(nn.Module):
    def __init__(self, embed_dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class StandardDecoderBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, dropout, qkv_bias, max_ctx):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, n_heads, bias=qkv_bias, dropout=dropout, max_ctx=max_ctx)
        self.ffn = StandardFeedForward(embed_dim, 4 * embed_dim, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class StandardGPT(nn.Module):
    def __init__(self, vocab_size, context_length, embed_dim, n_layers, n_heads, dropout, qkv_bias):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_length, embed_dim)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            StandardDecoderBlock(embed_dim, n_heads, dropout, qkv_bias, max_ctx=context_length) 
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.context_length = context_length

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device, dtype=torch.long).unsqueeze(0)  
        x = self.tok_emb(idx) + self.pos_emb(pos) 
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x) 
        return logits

# 2. Initialize the baseline model and load the original SFT weights
print("Loading Original SFT weights (gpt_sft_model.pt)... 💾")
baseline_model = StandardGPT(
    vocab_size=cfg.vocab_size, context_length=cfg.context_length, embed_dim=cfg.embed_dim,
    n_layers=cfg.n_layers, n_heads=cfg.n_heads, dropout=0.0, qkv_bias=cfg.qkv_bias
).to(device)

baseline_model.load_state_dict(torch.load("gpt_sft_model.pt", map_location=device))
baseline_model.eval()

# 3. Evaluate using our robust bAbI function
print("\nRunning bAbI Evaluation on Baseline SFT Model...")
baseline_babi_acc = evaluate_babi_robust(baseline_model, tok, device, num_samples=100)

print("\n" + "="*50)
print(" ⚖️ bAbI (LOGICAL REASONING) COMPARISON ⚖️")
print("="*50)
print(f"Baseline SFT Accuracy: {baseline_babi_acc:.2f}%")
try:
    print(f"Advanced MoE Accuracy: {babi_acc:.2f}%")
    print("-" * 50)
    print(f"MoE Improvement:       {babi_acc - baseline_babi_acc:+.2f}%")
except NameError:
    print("(Run the previous MoE bAbI cell to see the comparison delta)")
print("="*50)

--- Preparing Baseline SFT Model for bAbI Evaluation ---
Loading Original SFT weights (gpt_sft_model.pt)... 💾


/tmp/ipykernel_33580/3671923.py:64: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  baseline_model.load_state_dict(torch.load("gpt_sft_model.pt", map_location=device))



Running bAbI Evaluation on Baseline SFT Model...

--- Bypassing AWS 403: Fetching bAbI via Keras Mirror ---
Evaluating Model on bAbI Logical Reasoning... 🧠


 10%|████                                    | 300/3000 [01:45<15:52,  2.84it/s]


🏆 MoE bAbI Accuracy: 37.00%

 ⚖️ bAbI (LOGICAL REASONING) COMPARISON ⚖️
Baseline SFT Accuracy: 37.00%
Advanced MoE Accuracy: 36.00%
--------------------------------------------------
MoE Improvement:       -1.00%


In [44]:
import torch

print("--- Preparing Exact Baseline SFT Model for bAbI Evaluation ---")

# Check for available device
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Create an empty model using the exact parameter names your GPT class expects
# (Ensure this is run BEFORE the cell that redefines GPT with MoELayer)
baseline_model = GPT(
    vocab_size=cfg.vocab_size,
    context_length=cfg.context_length,
    embed_dim=cfg.embed_dim,
    n_heads=cfg.n_heads,       
    n_layers=cfg.n_layers,     
    dropout=cfg.dropout,       
    qkv_bias=cfg.qkv_bias,
).to(device)

# 2. Inject the trained weights from the saved file
baseline_model.load_state_dict(torch.load("gpt_sft_model.pt", map_location=device))

# 3. Set the model to evaluation mode
baseline_model.eval()

# 4. Evaluate using the robust bAbI function we wrote earlier
print("\nRunning bAbI Evaluation on Baseline SFT Model...")
# Note: Ensure evaluate_babi_robust and tok are defined in memory
baseline_babi_acc = evaluate_babi_robust(baseline_model, tok, device, num_samples=100)

print("\n" + "="*50)
print(" ⚖️ bAbI (LOGICAL REASONING) COMPARISON ⚖️")
print("="*50)
print(f"Baseline SFT Accuracy: {baseline_babi_acc:.2f}%")

# If the MoE result is in memory, it will print the delta
if 'babi_acc' in locals() or 'babi_acc' in globals():
    print(f"Advanced MoE Accuracy: {babi_acc:.2f}%")
    print("-" * 50)
    print(f"MoE Improvement:       {babi_acc - baseline_babi_acc:+.2f}%")
else:
    print("(Run the MoE evaluation later to compare results)")
print("="*50)

--- Preparing Exact Baseline SFT Model for bAbI Evaluation ---


TypeError: GPT.__init__() missing 2 required positional arguments: 'num_experts' and 'top_k'